In [7]:
import hashlib
import json
import uuid
from dataclasses import dataclass, field, asdict
from datetime import datetime, date, timezone
from enum import Enum
from typing import Optional

print('✅ Imports OK')

✅ Imports OK


In [8]:
class BatteryChemistry(Enum):
    """VDA 284 §5.1 — Approved cathode chemistry categories."""
    LFP        = "LiFePO4"          # Lithium Iron Phosphate
    NMC        = "LiNiMnCoO2"       # Lithium Nickel Manganese Cobalt
    NCA        = "LiNiCoAlO2"       # Lithium Nickel Cobalt Aluminium
    LCO        = "LiCoO2"           # Lithium Cobalt Oxide
    LMO        = "LiMn2O4"          # Lithium Manganese Oxide
    LNMO       = "LiNi0.5Mn1.5O4"  # High-voltage Spinel
    NiMH       = "NiMH"             # Nickel-Metal Hydride
    SolidState = "SolidState"       # Next-gen Solid-State


class BatteryApplication(Enum):
    """JTC 24 — Application categories for end-of-life routing."""
    EV         = "Electric Vehicle"
    HEV        = "Hybrid Electric Vehicle"
    ESS        = "Energy Storage System"
    INDUSTRIAL = "Industrial"
    PORTABLE   = "Portable"


class LifecycleState(Enum):
    """EU Battery Regulation — Lifecycle state machine."""
    MANUFACTURED = "Manufactured"
    IN_USE       = "In Use"
    SECOND_LIFE  = "Second Life / Repurposed"
    END_OF_LIFE  = "End of Life"
    RECYCLED     = "Recycled"


# Quick preview
print("Available chemistries:")
for c in BatteryChemistry:
    print(f"  BatteryChemistry.{c.name:12s} → {c.value}")

print("\nAvailable lifecycle states:")
for s in LifecycleState:
    print(f"  LifecycleState.{s.name:15s} → {s.value}")

Available chemistries:
  BatteryChemistry.LFP          → LiFePO4
  BatteryChemistry.NMC          → LiNiMnCoO2
  BatteryChemistry.NCA          → LiNiCoAlO2
  BatteryChemistry.LCO          → LiCoO2
  BatteryChemistry.LMO          → LiMn2O4
  BatteryChemistry.LNMO         → LiNi0.5Mn1.5O4
  BatteryChemistry.NiMH         → NiMH
  BatteryChemistry.SolidState   → SolidState

Available lifecycle states:
  LifecycleState.MANUFACTURED    → Manufactured
  LifecycleState.IN_USE          → In Use
  LifecycleState.SECOND_LIFE     → Second Life / Repurposed
  LifecycleState.END_OF_LIFE     → End of Life
  LifecycleState.RECYCLED        → Recycled


In [9]:
@dataclass
class VDA284_StrategicLayer:
    """
    VDA 284 Strategic Layer — Chemistry & Capacity
    Aligns with VDA 284 §4 (Identification) and §6 (Performance).
    """
    # ── Identification ────────────────────────────────────────────────────
    manufacturer_id:    str              # VDA 284 §4.1 — OEM / Cell maker
    model_designation:  str              # VDA 284 §4.2 — Commercial model name
    manufacturing_date: date             # VDA 284 §4.3 — ISO 8601 date

    # ── Chemistry (core strategic parameter) ──────────────────────────────
    chemistry:        BatteryChemistry   # VDA 284 §5.1 — Cathode chemistry
    anode_material:   str                # VDA 284 §5.2 — e.g. Graphite, Silicon
    electrolyte_type: str                # VDA 284 §5.3 — Liquid / Solid / Gel

    # ── Capacity & Energy (core strategic parameters) ─────────────────────
    nominal_capacity_ah: float           # VDA 284 §6.1 — Ampere-hours
    nominal_voltage_v:   float           # VDA 284 §6.2 — Volts
    energy_content_kwh:  float = field(init=False)  # Auto-calculated

    # ── Power & Thermal ───────────────────────────────────────────────────
    max_charge_rate_c:    float = 1.0    # VDA 284 §6.4 — C-rate
    max_discharge_rate_c: float = 2.0   # VDA 284 §6.5 — C-rate
    operating_temp_min_c: float = -20.0 # VDA 284 §6.6
    operating_temp_max_c: float =  60.0 # VDA 284 §6.6

    # ── Pack Architecture ─────────────────────────────────────────────────
    cell_count:       int   = 1          # VDA 284 §7.1
    series_strings:   int   = 1          # VDA 284 §7.2
    parallel_strings: int   = 1          # VDA 284 §7.3
    pack_mass_kg:     float = 0.0        # VDA 284 §7.4

    # ── SoH Thresholds ────────────────────────────────────────────────────
    soh_second_life_threshold_pct: float = 80.0  # Min SoH for 2nd-life
    soh_eol_threshold_pct:         float = 60.0  # SoH at which EOL declared

    def __post_init__(self):
        self.energy_content_kwh = round(
            (self.nominal_capacity_ah * self.nominal_voltage_v) / 1000, 4
        )
        self._validate()

    def _validate(self):
        assert self.nominal_capacity_ah > 0, "Capacity must be > 0 (VDA 284 §6.1)"
        assert self.nominal_voltage_v   > 0, "Voltage must be > 0 (VDA 284 §6.2)"
        assert self.cell_count >= 1,         "Cell count must be ≥ 1 (VDA 284 §7.1)"
        assert 0 < self.soh_eol_threshold_pct < self.soh_second_life_threshold_pct <= 100, \
            "SoH thresholds must satisfy: EOL < 2nd-Life ≤ 100"

    @property
    def specific_energy_wh_per_kg(self) -> Optional[float]:
        """Gravimetric energy density — key for second-life assessment."""
        if self.pack_mass_kg > 0:
            return round((self.energy_content_kwh * 1000) / self.pack_mass_kg, 2)
        return None

    @property
    def power_output_kw(self) -> float:
        """Peak discharge power."""
        return round(self.energy_content_kwh * self.max_discharge_rate_c, 2)

    def to_dict(self) -> dict:
        d = asdict(self)
        d['chemistry']                 = self.chemistry.value
        d['energy_content_kwh']        = self.energy_content_kwh
        d['specific_energy_wh_per_kg'] = self.specific_energy_wh_per_kg
        d['power_output_kw']           = self.power_output_kw
        d['manufacturing_date']        = self.manufacturing_date.isoformat()
        d['_standard']                 = 'VDA 284'
        d['_layer']                    = 'Strategic – Chemistry & Capacity'
        return d

print('✅ VDA284_StrategicLayer defined')

✅ VDA284_StrategicLayer defined


In [10]:
@dataclass
class JTC24_CircularityLayer:
    """
    JTC 24 Circularity Layer — Recycled Content & End-of-Life
    EU Battery Regulation Art. 8 (recycled content), Art. 11 (end-of-life).
    """
    # ── Recycled Content (core circularity parameters) ────────────────────
    recycled_cobalt_pct:    float = 0.0  # JTC 24 §3.1 | EU Art. 8
    recycled_lithium_pct:   float = 0.0  # JTC 24 §3.2 | EU Art. 8
    recycled_nickel_pct:    float = 0.0  # JTC 24 §3.3 | EU Art. 8
    recycled_lead_pct:      float = 0.0  # JTC 24 §3.4 | EU Art. 8 (Pb-acid)
    recycled_manganese_pct: float = 0.0  # JTC 24 §3.5

    # ── Carbon Footprint (EU Art. 7) ──────────────────────────────────────
    carbon_footprint_kg_co2_per_kwh: float = 0.0
    carbon_footprint_scope:          str   = 'Cradle-to-gate'  # ISO 14067

    # ── Supply Chain Due Diligence (EU Art. 72) ───────────────────────────
    cobalt_supply_chain_verified:  bool = False
    lithium_supply_chain_verified: bool = False
    conflict_minerals_free:        bool = False

    # ── End-of-Life & Repairability ───────────────────────────────────────
    application:              BatteryApplication = BatteryApplication.EV
    dismantling_time_minutes: float = 0.0   # JTC 24 §5
    recyclability_rate_pct:   float = 0.0   # JTC 24 §6 — target ≥ 70% by 2030
    recovery_efficiency_pct:  float = 0.0   # JTC 24 §6 — per EU Annex XII

    # ── Lifecycle State ───────────────────────────────────────────────────
    lifecycle_state:  LifecycleState = LifecycleState.MANUFACTURED
    current_soh_pct:  float = 100.0

    # EU recycled content thresholds (Art. 8, Annex X)
    _EU_THRESHOLDS: dict = field(default_factory=lambda: {
        'phase1_2027': {'Co': 16.0, 'Li':  6.0, 'Ni':  6.0, 'Pb': 85.0},
        'phase2_2031': {'Co': 26.0, 'Li': 12.0, 'Ni': 15.0, 'Pb': 85.0},
    })

    def __post_init__(self):
        self._validate()

    def _validate(self):
        pct_fields = [
            'recycled_cobalt_pct', 'recycled_lithium_pct', 'recycled_nickel_pct',
            'recycled_lead_pct', 'recycled_manganese_pct',
            'recyclability_rate_pct', 'recovery_efficiency_pct',
        ]
        for attr in pct_fields:
            val = getattr(self, attr)
            assert 0.0 <= val <= 100.0, f"{attr} must be 0–100% (JTC 24 §3)"
        assert 0.0 <= self.current_soh_pct <= 100.0, "SoH must be 0–100%"

    @property
    def overall_recycled_content_pct(self) -> float:
        """Weighted average recycled content across active critical materials."""
        values = [
            self.recycled_cobalt_pct, self.recycled_lithium_pct,
            self.recycled_nickel_pct, self.recycled_manganese_pct,
        ]
        active = [v for v in values if v > 0]
        return round(sum(active) / len(active), 2) if active else 0.0

    def check_eu_compliance(self) -> dict:
        """Evaluate compliance against EU Art. 8 targets per phase."""
        mapping = {
            'Co': self.recycled_cobalt_pct,
            'Li': self.recycled_lithium_pct,
            'Ni': self.recycled_nickel_pct,
            'Pb': self.recycled_lead_pct,
        }
        results = {}
        for phase, thresholds in self._EU_THRESHOLDS.items():
            results[phase] = {
                mat: {
                    'actual_pct':    mapping[mat],
                    'threshold_pct': thresh,
                    'compliant':     mapping[mat] >= thresh,
                    'gap_pct':       max(0.0, round(thresh - mapping[mat], 2)),
                }
                for mat, thresh in thresholds.items()
            }
        return results

    def recommend_lifecycle_action(
        self, soh_pct: float,
        second_life_threshold: float,
        eol_threshold: float
    ) -> str:
        """Smart circular economy routing based on State of Health."""
        if soh_pct >= second_life_threshold:
            return 'CONTINUE_PRIMARY_USE'
        elif soh_pct >= eol_threshold:
            return 'SECOND_LIFE_REPURPOSE'
        else:
            return 'END_OF_LIFE_RECYCLE'

    def to_dict(self) -> dict:
        d = asdict(self)
        d['application']                  = self.application.value
        d['lifecycle_state']              = self.lifecycle_state.value
        d['overall_recycled_content_pct'] = self.overall_recycled_content_pct
        d['eu_compliance_check']          = self.check_eu_compliance()
        d.pop('_EU_THRESHOLDS', None)
        d['_standard'] = 'JTC 24'
        d['_layer']    = 'Circularity – Recycled Content & End-of-Life'
        return d

print('✅ JTC24_CircularityLayer defined')

✅ JTC24_CircularityLayer defined


In [11]:
@dataclass
class SHA256_IntegrityLayer:
    """
    SHA-256 Integrity Layer — Tamper-evident cryptographic fingerprint.
    Hash is computed over canonical JSON (sort_keys=True) of both layers.
    Aligns with EU Battery Regulation Art. 65 (data authenticity).
    """
    passport_id:            str
    strategic_hash_input:   dict
    circularity_hash_input: dict
    timestamp_utc: str = field(
        default_factory=lambda: datetime.now(timezone.utc).isoformat()
    )

    # Auto-computed — do not set manually
    strategic_sha256:         str = field(init=False)
    circularity_sha256:       str = field(init=False)
    combined_passport_sha256: str = field(init=False)

    def __post_init__(self):
        self.strategic_sha256   = self._hash(self.strategic_hash_input)
        self.circularity_sha256 = self._hash(self.circularity_hash_input)
        combined = {
            'passport_id':        self.passport_id,
            'timestamp_utc':      self.timestamp_utc,
            'strategic_sha256':   self.strategic_sha256,
            'circularity_sha256': self.circularity_sha256,
        }
        self.combined_passport_sha256 = self._hash(combined)

    @staticmethod
    def _hash(data: dict) -> str:
        """Deterministic SHA-256: sorted keys, UTF-8 canonical JSON."""
        canonical = json.dumps(data, sort_keys=True, ensure_ascii=True, default=str)
        return hashlib.sha256(canonical.encode('utf-8')).hexdigest()

    def verify(self, strategic_data: dict, circularity_data: dict) -> dict:
        """Re-compute hashes and compare — detects any data tampering."""
        new_strat = self._hash(strategic_data)
        new_circ  = self._hash(circularity_data)
        strat_ok  = new_strat == self.strategic_sha256
        circ_ok   = new_circ  == self.circularity_sha256
        return {
            'strategic_layer_intact':   strat_ok,
            'circularity_layer_intact': circ_ok,
            'passport_intact':          strat_ok and circ_ok,
            'verification_timestamp':   datetime.now(timezone.utc).isoformat(),
        }

    def to_dict(self) -> dict:
        return {
            'passport_id':              self.passport_id,
            'timestamp_utc':            self.timestamp_utc,
            'strategic_sha256':         self.strategic_sha256,
            'circularity_sha256':       self.circularity_sha256,
            'combined_passport_sha256': self.combined_passport_sha256,
            '_algorithm':               'SHA-256 (FIPS 180-4)',
            '_encoding':                'Canonical JSON, sort_keys=True, UTF-8',
            '_layer':                   'Integrity – Tamper Evidence',
        }

print('✅ SHA256_IntegrityLayer defined')

✅ SHA256_IntegrityLayer defined


In [12]:
class BatteryPassport:
    """
    EU Battery Passport — Composite Root
    =====================================
    Integrates VDA 284 (Strategic) + JTC 24 (Circularity) + SHA-256 (Integrity).
    """

    def __init__(
        self,
        strategic:   VDA284_StrategicLayer,
        circularity: JTC24_CircularityLayer,
    ):
        self.passport_id  = str(uuid.uuid4())
        self.strategic    = strategic
        self.circularity  = circularity
        self.integrity    = SHA256_IntegrityLayer(
            passport_id            = self.passport_id,
            strategic_hash_input   = strategic.to_dict(),
            circularity_hash_input = circularity.to_dict(),
        )

    # ── Smart Circular Economy Functions ──────────────────────────────────

    def assess_second_life(self) -> dict:
        """Evaluate second-life candidacy using VDA 284 SoH thresholds."""
        soh    = self.circularity.current_soh_pct
        action = self.circularity.recommend_lifecycle_action(
            soh,
            self.strategic.soh_second_life_threshold_pct,
            self.strategic.soh_eol_threshold_pct,
        )
        destinations = {
            'CONTINUE_PRIMARY_USE':  self.circularity.application.value,
            'SECOND_LIFE_REPURPOSE': 'Stationary Energy Storage / Grid Buffer',
            'END_OF_LIFE_RECYCLE':   'Material Recovery & Recycling',
        }
        return {
            'current_soh_pct':           soh,
            'second_life_threshold_pct': self.strategic.soh_second_life_threshold_pct,
            'eol_threshold_pct':         self.strategic.soh_eol_threshold_pct,
            'recommended_action':        action,
            'recommended_application':   destinations[action],
            'remaining_energy_kwh':      round(
                self.strategic.energy_content_kwh * soh / 100, 4
            ),
        }

    def eu_compliance_report(self) -> dict:
        """Full EU Battery Regulation Art. 8 compliance snapshot."""
        compliance  = self.circularity.check_eu_compliance()
        phase1_pass = all(v['compliant'] for v in compliance['phase1_2027'].values())
        phase2_pass = all(v['compliant'] for v in compliance['phase2_2031'].values())
        return {
            'passport_id':              self.passport_id,
            'battery_model':            self.strategic.model_designation,
            'chemistry':                self.strategic.chemistry.value,
            'overall_recycled_content': f"{self.circularity.overall_recycled_content_pct}%",
            'phase1_2027_compliant':    phase1_pass,
            'phase2_2031_compliant':    phase2_pass,
            'detail':                   compliance,
        }

    def verify_integrity(self) -> dict:
        """SHA-256 cryptographic verification of passport data."""
        return self.integrity.verify(
            self.strategic.to_dict(),
            self.circularity.to_dict(),
        )

    def export_passport(self) -> dict:
        """Export the complete battery passport as a structured dictionary."""
        return {
            'passport_id':      self.passport_id,
            'regulation':       'EU Battery Regulation 2023/1542',
            'generated_at_utc': datetime.now(timezone.utc).isoformat(),
            'layers': {
                'strategic_layer_VDA284':  self.strategic.to_dict(),
                'circularity_layer_JTC24': self.circularity.to_dict(),
                'integrity_layer_SHA256':  self.integrity.to_dict(),
            },
            'smart_functions': {
                'second_life_assessment':  self.assess_second_life(),
                'eu_compliance_report':    self.eu_compliance_report(),
                'integrity_verification':  self.verify_integrity(),
            },
        }

print('✅ BatteryPassport defined — all classes ready')

✅ BatteryPassport defined — all classes ready


In [13]:
# ════════════════════════════════════════════════════════════════════
# 🔵  VDA 284 — STRATEGIC LAYER
# Edit these values to match your battery pack
# ════════════════════════════════════════════════════════════════════

strategic = VDA284_StrategicLayer(
    # Identification
    manufacturer_id    = "ACME-Cells-GmbH",
    model_designation  = "EV-NMC-75kWh-Gen2",
    manufacturing_date = date(2024, 3, 15),

    # Chemistry — choose from:
    # BatteryChemistry.NMC / .LFP / .NCA / .LCO / .LMO / .NiMH / .SolidState
    chemistry        = BatteryChemistry.NMC,
    anode_material   = "Graphite / 5% Silicon",
    electrolyte_type = "Liquid Carbonate (LP57)",

    # Capacity & Energy — energy_content_kwh is calculated automatically
    # 200 Ah × 375 V ÷ 1000 = 75.0 kWh
    nominal_capacity_ah = 200.0,
    nominal_voltage_v   = 375.0,

    # Power & Thermal
    max_charge_rate_c    = 1.5,
    max_discharge_rate_c = 3.0,
    operating_temp_min_c = -30.0,
    operating_temp_max_c =  55.0,

    # Pack Architecture
    cell_count       = 288,
    series_strings   = 96,
    parallel_strings = 3,
    pack_mass_kg     = 450.0,

    # SoH Thresholds — these drive the Cell 9 routing decision
    soh_second_life_threshold_pct = 80.0,  # ≥ 80% → still in primary use
    soh_eol_threshold_pct         = 60.0,  # < 60% → end of life
)

print(f"✅ Strategic layer configured")
print(f"   Chemistry      : {strategic.chemistry.value}")
print(f"   Energy         : {strategic.energy_content_kwh} kWh")
print(f"   Peak Power     : {strategic.power_output_kw} kW")
print(f"   Specific Energy: {strategic.specific_energy_wh_per_kg} Wh/kg")

✅ Strategic layer configured
   Chemistry      : LiNiMnCoO2
   Energy         : 75.0 kWh
   Peak Power     : 225.0 kW
   Specific Energy: 166.67 Wh/kg


In [14]:
# ════════════════════════════════════════════════════════════════════
# 🟢  JTC 24 — CIRCULARITY LAYER
# Edit recycled content percentages and lifecycle data
# ════════════════════════════════════════════════════════════════════

circularity = JTC24_CircularityLayer(
    # Recycled Content (%) — EU Art. 8 mandatory thresholds:
    # Phase 1 (2027): Co ≥ 16%   Li ≥ 6%   Ni ≥ 6%   Pb ≥ 85%
    # Phase 2 (2031): Co ≥ 26%   Li ≥ 12%  Ni ≥ 15%  Pb ≥ 85%
    recycled_cobalt_pct    = 18.0,   # ✅ Phase 1 | ❌ Phase 2
    recycled_lithium_pct   =  8.5,   # ✅ Phase 1 | ❌ Phase 2
    recycled_nickel_pct    =  9.0,   # ✅ Phase 1 | ❌ Phase 2
    recycled_lead_pct      =  0.0,   # N/A for NMC chemistry
    recycled_manganese_pct = 12.0,

    # Carbon Footprint (EU Art. 7)
    carbon_footprint_kg_co2_per_kwh = 65.0,
    carbon_footprint_scope          = "Cradle-to-gate (ISO 14067)",

    # Supply Chain Due Diligence (EU Art. 72)
    cobalt_supply_chain_verified  = True,
    lithium_supply_chain_verified = True,
    conflict_minerals_free        = True,

    # End-of-Life
    application              = BatteryApplication.EV,
    dismantling_time_minutes = 45.0,
    recyclability_rate_pct   = 72.0,  # target ≥ 70% by 2030 ✅
    recovery_efficiency_pct  = 68.0,

    # Current State — try changing this to 75 or 55 to see different routing
    lifecycle_state  = LifecycleState.IN_USE,
    current_soh_pct  = 90.0,
)

print(f"✅ Circularity layer configured")
print(f"   Overall Recycled Content : {circularity.overall_recycled_content_pct}%")
print(f"   Current SoH              : {circularity.current_soh_pct}%")
print(f"   Lifecycle State          : {circularity.lifecycle_state.value}")

✅ Circularity layer configured
   Overall Recycled Content : 11.88%
   Current SoH              : 90.0%
   Lifecycle State          : In Use


In [15]:
# ════════════════════════════════════════════════════════════════════
# 🏗️  BUILD THE PASSPORT
# One line — SHA-256 hashes are computed automatically
# ════════════════════════════════════════════════════════════════════

passport = BatteryPassport(strategic, circularity)

print(f"🔋 Battery Passport created")
print(f"   Passport ID : {passport.passport_id}")
print(f"   Model       : {passport.strategic.model_designation}")
print(f"   Chemistry   : {passport.strategic.chemistry.value}")
print()
print(f"   🔴 SHA-256 Hashes")
print(f"   Strategic   : {passport.integrity.strategic_sha256}")
print(f"   Circularity : {passport.integrity.circularity_sha256}")
print(f"   Combined    : {passport.integrity.combined_passport_sha256}")

🔋 Battery Passport created
   Passport ID : 5d2205da-d23e-4198-b526-fca2ce95b035
   Model       : EV-NMC-75kWh-Gen2
   Chemistry   : LiNiMnCoO2

   🔴 SHA-256 Hashes
   Strategic   : a49f8a49912345a4b0cde6734130213531c80ee881efb29c375cb4de8a2453f2
   Circularity : 9244d05c138f543ed5a155472d3910034064b780d3cb8533b092d1be25f9c485
   Combined    : eebdedde45612843cb769586379c8a0e6439de45ab97be22b2d2f55033777aee


In [16]:
# ════════════════════════════════════════════════════════════════════
# ♻️  SMART FUNCTION 1 — SECOND-LIFE ASSESSMENT
# Routes the battery based on current State of Health
# ════════════════════════════════════════════════════════════════════

sl = passport.assess_second_life()

action_icons = {
    'CONTINUE_PRIMARY_USE':  '🚗  Still performing — keep in service',
    'SECOND_LIFE_REPURPOSE': '🏭  Repurpose for stationary storage',
    'END_OF_LIFE_RECYCLE':   '♻️  Send for material recovery',
}

print("━" * 55)
print("  ♻️  SECOND-LIFE ASSESSMENT")
print("━" * 55)
print(f"  Current SoH            : {sl['current_soh_pct']}%")
print(f"  2nd-Life Threshold     : ≥ {sl['second_life_threshold_pct']}%")
print(f"  EOL Threshold          : < {sl['eol_threshold_pct']}%")
print(f"  Remaining Energy       : {sl['remaining_energy_kwh']} kWh")
print()
print(f"  Recommended Action     : {sl['recommended_action']}")
print(f"  {action_icons[sl['recommended_action']]}")
print(f"  Recommended Use        : {sl['recommended_application']}")
print("━" * 55)
print()
print("💡 Tip: Change current_soh_pct in Cell 7b then re-run")
print("        84.5 → CONTINUE  |  75.0 → SECOND LIFE  |  55.0 → EOL")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ♻️  SECOND-LIFE ASSESSMENT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Current SoH            : 90.0%
  2nd-Life Threshold     : ≥ 80.0%
  EOL Threshold          : < 60.0%
  Remaining Energy       : 67.5 kWh

  Recommended Action     : CONTINUE_PRIMARY_USE
  🚗  Still performing — keep in service
  Recommended Use        : Electric Vehicle
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

💡 Tip: Change current_soh_pct in Cell 7b then re-run
        84.5 → CONTINUE  |  75.0 → SECOND LIFE  |  55.0 → EOL


In [17]:
# ════════════════════════════════════════════════════════════════════
# 🇪🇺  SMART FUNCTION 2 — EU COMPLIANCE REPORT
# Checks recycled content against Phase 1 (2027) and Phase 2 (2031)
# ════════════════════════════════════════════════════════════════════

comp = passport.eu_compliance_report()

print("━" * 60)
print("  🇪🇺  EU BATTERY REGULATION ART. 8 — COMPLIANCE REPORT")
print("━" * 60)
print(f"  Model           : {comp['battery_model']}")
print(f"  Chemistry       : {comp['chemistry']}")
print(f"  Overall Recycled: {comp['overall_recycled_content']}")
print()

for phase, label in [('phase1_2027', 'Phase 1 — 2027'),
                     ('phase2_2031', 'Phase 2 — 2031')]:

    phase_pass = all(v['compliant'] for v in comp['detail'][phase].values())
    status     = '✅ PASS' if phase_pass else '❌ FAIL'

    print(f"  {label}  [{status}]")
    print(f"  {'Material':<12} {'Actual':>8}  {'Target':>8}  {'Gap':>8}  Status")
    print(f"  {'-'*52}")

    for mat, data in comp['detail'][phase].items():
        tick = '✅' if data['compliant'] else '❌'
        gap  = f"-{data['gap_pct']}%" if data['gap_pct'] > 0 else '—'
        print(f"  {mat:<12} {data['actual_pct']:>7}%  "
              f"{data['threshold_pct']:>7}%  {gap:>8}  {tick}")
    print()

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  🇪🇺  EU BATTERY REGULATION ART. 8 — COMPLIANCE REPORT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Model           : EV-NMC-75kWh-Gen2
  Chemistry       : LiNiMnCoO2
  Overall Recycled: 11.88%

  Phase 1 — 2027  [❌ FAIL]
  Material       Actual    Target       Gap  Status
  ----------------------------------------------------
  Co              18.0%     16.0%         —  ✅
  Li               8.5%      6.0%         —  ✅
  Ni               9.0%      6.0%         —  ✅
  Pb               0.0%     85.0%    -85.0%  ❌

  Phase 2 — 2031  [❌ FAIL]
  Material       Actual    Target       Gap  Status
  ----------------------------------------------------
  Co              18.0%     26.0%     -8.0%  ❌
  Li               8.5%     12.0%     -3.5%  ❌
  Ni               9.0%     15.0%     -6.0%  ❌
  Pb               0.0%     85.0%    -85.0%  ❌



In [18]:
# ════════════════════════════════════════════════════════════════════
# 🔐  SMART FUNCTION 3 — SHA-256 INTEGRITY VERIFICATION
# Re-computes hashes and compares to stored fingerprints
# ════════════════════════════════════════════════════════════════════

intg = passport.verify_integrity()

print("━" * 55)
print("  🔐  SHA-256 INTEGRITY VERIFICATION")
print("━" * 55)
print(f"  Strategic Layer intact   : "
      f"{'✅ YES' if intg['strategic_layer_intact']   else '❌ TAMPERED'}")
print(f"  Circularity Layer intact : "
      f"{'✅ YES' if intg['circularity_layer_intact'] else '❌ TAMPERED'}")
print(f"  Passport intact          : "
      f"{'✅ VERIFIED' if intg['passport_intact']     else '❌ TAMPERED'}")
print(f"  Verified at              : {intg['verification_timestamp']}")
print()
print("  Hashes on record:")
print(f"  Strategic   : {passport.integrity.strategic_sha256}")
print(f"  Circularity : {passport.integrity.circularity_sha256}")
print(f"  Combined    : {passport.integrity.combined_passport_sha256}")
print("━" * 55)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  🔐  SHA-256 INTEGRITY VERIFICATION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Strategic Layer intact   : ✅ YES
  Circularity Layer intact : ✅ YES
  Passport intact          : ✅ VERIFIED
  Verified at              : 2026-04-15T11:50:42.776655+00:00

  Hashes on record:
  Strategic   : a49f8a49912345a4b0cde6734130213531c80ee881efb29c375cb4de8a2453f2
  Circularity : 9244d05c138f543ed5a155472d3910034064b780d3cb8533b092d1be25f9c485
  Combined    : eebdedde45612843cb769586379c8a0e6439de45ab97be22b2d2f55033777aee
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


In [19]:
# ════════════════════════════════════════════════════════════════════
# 🔬  TAMPER DEMONSTRATION
# Shows SHA-256 catching a falsified recycled content value
# ════════════════════════════════════════════════════════════════════

print("Simulating falsified data...")
print("  Original recycled_cobalt_pct : 18.0%")
print("  Falsified recycled_cobalt_pct: 30.0%  ← inflated to pass Phase 2")
print()

# Build a fake circularity layer with the inflated cobalt figure
tampered_circularity = JTC24_CircularityLayer(
    recycled_cobalt_pct    = 30.0,   # ← FALSIFIED (was 18.0)
    recycled_lithium_pct   =  8.5,   # everything else unchanged
    recycled_nickel_pct    =  15.0,
    recycled_lead_pct      =  0.0,
    recycled_manganese_pct = 12.0,
    carbon_footprint_kg_co2_per_kwh = 65.0,
    application             = BatteryApplication.EV,
    lifecycle_state         = LifecycleState.IN_USE,
    current_soh_pct         = 84.5,
)

# Run the ORIGINAL passport's verify() against the falsified data
# The passport still holds the hash of the REAL 18.0% figure
tamper_result = passport.integrity.verify(
    passport.strategic.to_dict(),       # strategic unchanged — should pass
    tampered_circularity.to_dict(),     # falsified — should fail
)

print("━" * 55)
print("  🔬  VERIFICATION RESULT WITH FALSIFIED DATA")
print("━" * 55)
print(f"  Strategic Layer intact   : "
      f"{'✅ YES' if tamper_result['strategic_layer_intact']   else '❌ TAMPERED'}")
print(f"  Circularity Layer intact : "
      f"{'✅ YES' if tamper_result['circularity_layer_intact'] else '❌ TAMPERED'}")
print(f"  Passport intact          : "
      f"{'✅ VERIFIED' if tamper_result['passport_intact']     else '❌ TAMPERED'}")
print()
print("  Hash on record (original 18.0%):")
print(f"  {passport.integrity.circularity_sha256}")
print()
print("  Hash of falsified data (30.0%):")
print(f"  {SHA256_IntegrityLayer._hash(tampered_circularity.to_dict())}")
print("━" * 55)
print()
print("→ SHA-256 correctly catches the falsified recycled cobalt content.")
print("→ The strategic layer remains ✅ — only the circularity layer was touched.")

Simulating falsified data...
  Original recycled_cobalt_pct : 18.0%
  Falsified recycled_cobalt_pct: 30.0%  ← inflated to pass Phase 2

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  🔬  VERIFICATION RESULT WITH FALSIFIED DATA
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Strategic Layer intact   : ✅ YES
  Circularity Layer intact : ❌ TAMPERED
  Passport intact          : ❌ TAMPERED

  Hash on record (original 18.0%):
  9244d05c138f543ed5a155472d3910034064b780d3cb8533b092d1be25f9c485

  Hash of falsified data (30.0%):
  f8813c6b161a89b256eba2e6b03606c1f4d188c3cb18d2381ae3ee8a4cf2926d
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

→ SHA-256 correctly catches the falsified recycled cobalt content.
→ The strategic layer remains ✅ — only the circularity layer was touched.


In [20]:
# ════════════════════════════════════════════════════════════════════
# 💾  EXPORT FULL PASSPORT TO JSON
# All three layers + all smart function results in one file
# ════════════════════════════════════════════════════════════════════

passport_data = passport.export_passport()

output_file = "battery_passport_export.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(passport_data, f, indent=2, default=str)

print(f"✅ Passport exported → {output_file}")
print(f"   Passport ID    : {passport_data['passport_id']}")
print(f"   Generated at   : {passport_data['generated_at_utc']}")
print(f"   Regulation     : {passport_data['regulation']}")
print()

# Preview the top-level structure
print("Top-level structure:")
for key in passport_data.keys():
    print(f"  {key}")

print("\nLayers included:")
for layer in passport_data['layers'].keys():
    print(f"  {layer}")

print("\nSmart functions included:")
for fn in passport_data['smart_functions'].keys():
    print(f"  {fn}")

print()
print("━" * 55)
print("  🔋  NOTEBOOK COMPLETE")
print("━" * 55)
print("  Next steps:")
print("  • Change current_soh_pct in Cell 7b → re-run 8–13")
print("  • Raise recycled_nickel_pct to 15 → re-run 10")
print("  • Swap chemistry to BatteryChemistry.LFP → re-run all")
print("  • Load the JSON and re-verify hashes independently")
print("━" * 55)

✅ Passport exported → battery_passport_export.json
   Passport ID    : 5d2205da-d23e-4198-b526-fca2ce95b035
   Generated at   : 2026-04-15T11:50:44.181717+00:00
   Regulation     : EU Battery Regulation 2023/1542

Top-level structure:
  passport_id
  regulation
  generated_at_utc
  layers
  smart_functions

Layers included:
  strategic_layer_VDA284
  circularity_layer_JTC24
  integrity_layer_SHA256

Smart functions included:
  second_life_assessment
  eu_compliance_report
  integrity_verification

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  🔋  NOTEBOOK COMPLETE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Next steps:
  • Change current_soh_pct in Cell 7b → re-run 8–13
  • Raise recycled_nickel_pct to 15 → re-run 10
  • Swap chemistry to BatteryChemistry.LFP → re-run all
  • Load the JSON and re-verify hashes independently
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


---

# 🤖 Autonomous Life-Cycle Arbitrator (ALA)

**Extends the EU Battery Passport with dynamic arbitration logic.**

| Module | What it does |
|--------|-------------|
| ALA Cell 1 | Imports for the arbitrator |
| ALA Cell 2 | Grid region registry — carbon intensity by country |
| ALA Cell 3 | `DynamicCarbonTracker` — lifetime carbon accumulator |
| ALA Cell 4 | Simulate a charging history |
| ALA Cell 5 | `RecyclerBid` and `SecondLifeBid` dataclasses |
| ALA Cell 6 | `CircularScore` — weighted multi-criteria scoring |
| ALA Cell 7 | `LifeCycleArbitrator` — `negotiate_retirement()` |
| ALA Cell 8 | `PrivacyPreservingCert` — safety commitment scheme |
| ALA Cell 9 | `ArbitrationReport` — full output and JSON export |
| ALA Cell 10 | Run the full arbitration on our NMC pack |

---
> All classes below depend on `passport`, `strategic`, and `circularity`  
> being defined above. Always run Kernel → Restart & Run All before starting.

In [21]:
# ══════════════════════════════════════════════════════════════════════
# ALA CELL 1 — Imports
# All passport classes are already in memory from the cells above.
# This cell adds only what the arbitrator needs on top.
# ══════════════════════════════════════════════════════════════════════

import hashlib
import json
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from typing import Any, ClassVar, Optional

print("✅ ALA imports OK")
print(f"   Passport already loaded — ID: {passport.passport_id[:16]}...")
print(f"   Chemistry  : {passport.strategic.chemistry.value}")
print(f"   Energy     : {passport.strategic.energy_content_kwh} kWh")
print(f"   Current SoH: {passport.circularity.current_soh_pct}%")

✅ ALA imports OK
   Passport already loaded — ID: 5d2205da-d23e-41...
   Chemistry  : LiNiMnCoO2
   Energy     : 75.0 kWh
   Current SoH: 90.0%


In [22]:
# ══════════════════════════════════════════════════════════════════════
# ALA CELL 2 — Grid Region Registry
# ══════════════════════════════════════════════════════════════════════
# Parameterised: every intensity value is a float you can overwrite.
# To connect a live feed:
#   GRID["DE"].intensity_g_per_kwh = live_api.get_intensity("DE")
# Source: Ember Climate / IEA 2023 electricity emissions data
# ══════════════════════════════════════════════════════════════════════

@dataclass
class GridRegion:
    """
    A named electricity grid with a carbon intensity value.

    intensity_g_per_kwh: grams of CO2-equivalent per kWh delivered.
    This is the single number you swap out for a live API value
    when moving from simulation to production.
    """
    code:                str
    name:                str
    intensity_g_per_kwh: float


# ── Reference regions (2023 annual averages) ──────────────────────────
# Overwrite any value with a live figure without touching anything else.

GRID: dict[str, GridRegion] = {
    "NO": GridRegion("NO", "Norway",        26.0),
    "SE": GridRegion("SE", "Sweden",        45.0),
    "FR": GridRegion("FR", "France",        56.0),
    "GB": GridRegion("GB", "UK",           148.0),
    "ES": GridRegion("ES", "Spain",        160.0),
    "NL": GridRegion("NL", "Netherlands",  290.0),
    "DE": GridRegion("DE", "Germany",      380.0),
    "PL": GridRegion("PL", "Poland",       720.0),
    "EU": GridRegion("EU", "EU Average",   255.0),
    "GL": GridRegion("GL", "Global Avg",   436.0),
}


# ── Preview ───────────────────────────────────────────────────────────
print("Grid carbon intensity registry")
print(f"{'Code':<6} {'Region':<16} {'gCO2/kWh':>10}  Scale")
print("─" * 55)
max_intensity = max(r.intensity_g_per_kwh for r in GRID.values())
for code, region in GRID.items():
    bar_len  = int((region.intensity_g_per_kwh / max_intensity) * 30)
    bar      = "█" * bar_len
    print(f"  {code:<4} {region.name:<16} {region.intensity_g_per_kwh:>8.0f}  {bar}")

print()
print(f"💡 Our battery is modelled in Germany → {GRID['DE'].intensity_g_per_kwh} gCO2/kWh")
print(f"   Cleanest grid : {min(GRID.values(), key=lambda r: r.intensity_g_per_kwh).name}")
print(f"   Heaviest grid : {max(GRID.values(), key=lambda r: r.intensity_g_per_kwh).name}")

Grid carbon intensity registry
Code   Region             gCO2/kWh  Scale
───────────────────────────────────────────────────────
  NO   Norway                 26  █
  SE   Sweden                 45  █
  FR   France                 56  ██
  GB   UK                    148  ██████
  ES   Spain                 160  ██████
  NL   Netherlands           290  ████████████
  DE   Germany               380  ███████████████
  PL   Poland                720  ██████████████████████████████
  EU   EU Average            255  ██████████
  GL   Global Avg            436  ██████████████████

💡 Our battery is modelled in Germany → 380.0 gCO2/kWh
   Cleanest grid : Norway
   Heaviest grid : Poland


In [23]:
# ══════════════════════════════════════════════════════════════════════
# ALA CELL 2B — Live Grid Carbon Intensity
# ══════════════════════════════════════════════════════════════════════
# Connects to Electricity Maps API to replace hardcoded grid intensity
# values with real-time carbon data.
#
# Toggle:
#   USE_LIVE_GRID_DATA = False  → use hardcoded GRID values (default)
#   USE_LIVE_GRID_DATA = True   → fetch live data from Electricity Maps
#
# API documentation: https://static.electricitymaps.com/api/docs/index.html
# Free sandbox tier: mirrors production structure 1:1
#
# IMPORTANT: Never commit your API key to a public GitHub repository.
# Store it in a separate config file added to .gitignore
# ══════════════════════════════════════════════════════════════════════

import requests
from datetime import datetime, timezone

# ── Configuration ──────────────────────────────────────────────────────────

USE_LIVE_GRID_DATA = True   # ← flip this to False to use hardcoded values

ELECTRICITY_MAPS_API_KEY = "TYPE YOUR API KEY HERE"

# Electricity Maps zone codes → your GRID keys
# Full zone list: https://static.electricitymaps.com/api/docs/index.html
ZONE_MAP = {
    "NO": "NO",    # Norway
    "FR": "FR",    # France
    "DE": "DE",    # Germany
    "PL": "PL",    # Poland
    "NL": "NL",    # Netherlands
    "ES": "ES",    # Spain
    "SE": "SE",    # Sweden
    "GB": "GB",    # United Kingdom — Electricity Maps uses "GB"
}

# ── API function ───────────────────────────────────────────────────────────

def fetch_live_intensity(zone_code: str, api_key: str) -> Optional[float]:
    """
    Fetch real-time carbon intensity for a grid zone from Electricity Maps.

    Args:
        zone_code:  Electricity Maps zone code (e.g. "NO", "DE", "GB")
        api_key:    Your Electricity Maps API key

    Returns:
        Carbon intensity in gCO2eq/kWh, or None if the request fails.

    In production: call this every hour to keep GRID values current.
    The sandbox returns realistic sample data mirroring production 1:1.
    """
    url = "https://api.electricitymaps.com/v3/carbon-intensity/latest"
    headers = {"auth-token": api_key}
    params  = {"zone": zone_code}

    try:
        response = requests.get(url, headers=headers, params=params, timeout=5)

        if response.status_code == 200:
            data = response.json()
            return float(data["carbonIntensity"])

        elif response.status_code == 401:
            print(f"    ⚠️  API key rejected for zone {zone_code} — check your key")
            return None

        elif response.status_code == 429:
            print(f"    ⚠️  Rate limit hit for zone {zone_code} — using fallback")
            return None

        else:
            print(f"    ⚠️  Unexpected status {response.status_code} for {zone_code}")
            return None

    except requests.exceptions.Timeout:
        print(f"    ⚠️  Timeout fetching {zone_code} — using fallback value")
        return None

    except requests.exceptions.ConnectionError:
        print(f"    ⚠️  No internet connection — using fallback value for {zone_code}")
        return None

    except Exception as e:
        print(f"    ⚠️  Unexpected error for {zone_code}: {e} — using fallback")
        return None


# ── Update GRID dictionary ─────────────────────────────────────────────────

def update_grid_with_live_data(api_key: str) -> dict[str, float]:
    """
    Fetch live intensity for all zones and update the GRID dictionary.

    Returns a dict of {grid_code: intensity} showing what was updated.
    Falls back to hardcoded value if any individual zone fetch fails.
    """
    updated = {}
    fallback = {}

    print(f"  Fetching live grid intensity from Electricity Maps...")
    print(f"  Timestamp: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')}")
    print()

    for grid_code, zone_code in ZONE_MAP.items():
        live_intensity = fetch_live_intensity(zone_code, api_key)

        if live_intensity is not None:
            old_value = GRID[grid_code].intensity_g_per_kwh

            # Update the GridRegion — replace intensity with live value
            GRID[grid_code] = GridRegion(
                code                = GRID[grid_code].code,
                name                = GRID[grid_code].name,
                intensity_g_per_kwh = live_intensity,
            )
            updated[grid_code] = live_intensity

            print(f"  ✅ {grid_code}  {GRID[grid_code].name:<16} "
                  f"{old_value:>6.1f} → {live_intensity:>6.1f} gCO2/kWh  (live)")
        else:
            fallback[grid_code] = GRID[grid_code].intensity_g_per_kwh
            print(f"  ↩️  {grid_code}  {GRID[grid_code].name:<16} "
                  f"{GRID[grid_code].intensity_g_per_kwh:>6.1f} gCO2/kWh  (fallback)")

    return updated


# ── Execute ────────────────────────────────────────────────────────────────

print("━" * 60)
print("  GRID CARBON INTENSITY")
print("━" * 60)

if USE_LIVE_GRID_DATA:
    print("  Mode: LIVE  (Electricity Maps API)")
    print()
    updated_zones = update_grid_with_live_data(ELECTRICITY_MAPS_API_KEY)
    print()
    print(f"  Updated  : {len(updated_zones)} zones with live data")
    print(f"  Fallback : {len(ZONE_MAP) - len(updated_zones)} zones using hardcoded values")
    print()
    print("  Current GRID state:")
    for code, region in GRID.items():
        print(f"    {code}  {region.name:<16}  {region.intensity_g_per_kwh:>6.1f} gCO2/kWh")
else:
    print("  Mode: HARDCODED  (set USE_LIVE_GRID_DATA = True for live data)")
    print()
    print("  Current GRID state:")
    for code, region in GRID.items():
        print(f"    {code}  {region.name:<16}  {region.intensity_g_per_kwh:>6.1f} gCO2/kWh")

print("━" * 60)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  GRID CARBON INTENSITY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Mode: LIVE  (Electricity Maps API)

  Fetching live grid intensity from Electricity Maps...
  Timestamp: 2026-04-15 11:50 UTC

  ✅ NO  Norway             26.0 →   75.0 gCO2/kWh  (live)
  ✅ FR  France             56.0 →   22.0 gCO2/kWh  (live)
  ✅ DE  Germany           380.0 →  288.0 gCO2/kWh  (live)
  ✅ PL  Poland            720.0 →  546.0 gCO2/kWh  (live)
  ✅ NL  Netherlands       290.0 →   99.0 gCO2/kWh  (live)
  ✅ ES  Spain             160.0 →   86.0 gCO2/kWh  (live)
  ✅ SE  Sweden             45.0 →   41.0 gCO2/kWh  (live)
  ✅ GB  UK                148.0 →   94.0 gCO2/kWh  (live)

  Updated  : 8 zones with live data
  Fallback : 0 zones using hardcoded values

  Current GRID state:
    NO  Norway              75.0 gCO2/kWh
    SE  Sweden              41.0 gCO2/kWh
    FR  France              22.0 gCO2/kWh
    GB  UK                  94.

In [24]:
# ══════════════════════════════════════════════════════════════════════
# ALA CELL 2C — DataValidationAgent
# ══════════════════════════════════════════════════════════════════════
# Validates all battery data before any agent acts on it.
#
# Why this matters:
#   The arbitration system trusts its inputs completely.
#   Bad data — sensor errors, data entry mistakes, implausible readings
#   — flows silently through the pipeline and corrupts the decision.
#   This agent catches those problems explicitly before they cause harm.
#
# Two levels of finding:
#   ERROR   — hard failure, blocks arbitration entirely
#             e.g. SoH of 150% (physically impossible)
#
#   WARNING — soft flag, allows arbitration but marks for human review
#             e.g. SoH of 99% (unusual, verify BMS calibration)
#
# In production:
#   Runs automatically every time a new battery is registered
#   or a charging session is recorded. Integrates with BMS
#   data quality flags and Catena-X data sovereignty checks.
# ══════════════════════════════════════════════════════════════════════


@dataclass
class ValidationFinding:
    """A single error or warning produced by the DataValidationAgent."""
    level:    str
    field:    str
    message:  str
    value:    Any

    def __str__(self) -> str:
        icon = "❌" if self.level == "ERROR" else "⚠️ "
        return f"  {icon} [{self.level}] {self.field}: {self.message} (got: {self.value})"


@dataclass
class ValidationReport:
    """
    The complete output of a DataValidationAgent.validate() call.

    passed:    True only when there are zero ERROR-level findings.
               WARNING findings do not block arbitration.
    findings:  All errors and warnings found, in order of severity.
    """
    passed:   bool
    findings: list[ValidationFinding]

    @property
    def errors(self) -> list[ValidationFinding]:
        return [f for f in self.findings if f.level == "ERROR"]

    @property
    def warnings(self) -> list[ValidationFinding]:
        return [f for f in self.findings if f.level == "WARNING"]

    @property
    def error_count(self) -> int:
        return len(self.errors)

    @property
    def warning_count(self) -> int:
        return len(self.warnings)

    def print_report(self) -> None:
        status = "✅ PASSED" if self.passed else "❌ FAILED"
        print("━" * 62)
        print("  DATA VALIDATION REPORT")
        print("━" * 62)
        print(f"  Result   : {status}")
        print(f"  Errors   : {self.error_count}")
        print(f"  Warnings : {self.warning_count}")

        if not self.findings:
            print()
            print("  All checks passed — no issues found.")
        else:
            print()
            print("  Findings:")
            for finding in self.findings:
                print(str(finding))

        if not self.passed:
            print()
            print("  ⛔ Arbitration blocked — resolve all ERRORs before proceeding.")
        elif self.warning_count > 0:
            print()
            print("  ⚠️  Arbitration allowed — review WARNINGs before executing decision.")

        print("━" * 62)

    def to_dict(self) -> dict:
        return {
            "passed":        self.passed,
            "error_count":   self.error_count,
            "warning_count": self.warning_count,
            "findings": [
                {
                    "level":   f.level,
                    "field":   f.field,
                    "message": f.message,
                    "value":   str(f.value),
                }
                for f in self.findings
            ],
        }


class DataValidationAgent:
    """
    Agent 0 — The Gatekeeper.

    Validates BatteryPassport data before any other agent acts on it.
    Runs a suite of checks across three categories:

      1. Physical plausibility  — values within physically possible ranges
      2. Regulatory compliance  — values meeting EU Battery Reg thresholds
      3. Internal consistency   — values that must logically relate to each other

    Usage:
        validator = DataValidationAgent()
        report    = validator.validate(passport)

        if not report.passed:
            report.print_report()
            raise ValueError("Battery data failed validation — arbitration blocked")
    """

    def validate(self, passport: BatteryPassport) -> ValidationReport:
        """
        Run all validation checks on a BatteryPassport.
        Returns a ValidationReport — never raises an exception itself.
        The caller decides what to do with errors and warnings.
        """
        findings: list[ValidationFinding] = []
        s = passport.strategic
        c = passport.circularity

        # ── Category 1: Physical plausibility ─────────────────────────

        # SoH must be 0–100%
        if not (0.0 <= c.current_soh_pct <= 100.0):
            findings.append(ValidationFinding(
                level   = "ERROR",
                field   = "current_soh_pct",
                message = "SoH must be between 0% and 100% — physically impossible value",
                value   = c.current_soh_pct,
            ))
        elif c.current_soh_pct > 99.0:
            findings.append(ValidationFinding(
                level   = "WARNING",
                field   = "current_soh_pct",
                message = "SoH above 99% is unusual — verify BMS calibration",
                value   = c.current_soh_pct,
            ))

        # Energy content must be positive
        if s.energy_content_kwh <= 0:
            findings.append(ValidationFinding(
                level   = "ERROR",
                field   = "energy_content_kwh",
                message = "Energy content must be greater than zero",
                value   = s.energy_content_kwh,
            ))

        # Nominal capacity must be positive
        if s.nominal_capacity_ah <= 0:
            findings.append(ValidationFinding(
                level   = "ERROR",
                field   = "nominal_capacity_ah",
                message = "Nominal capacity must be greater than zero",
                value   = s.nominal_capacity_ah,
            ))

        # Nominal voltage must be positive
        if s.nominal_voltage_v <= 0:
            findings.append(ValidationFinding(
                level   = "ERROR",
                field   = "nominal_voltage_v",
                message = "Nominal voltage must be greater than zero",
                value   = s.nominal_voltage_v,
            ))

        # Carbon footprint must be positive
        if c.carbon_footprint_kg_co2_per_kwh <= 0:
            findings.append(ValidationFinding(
                level   = "ERROR",
                field   = "carbon_footprint_kg_co2_per_kwh",
                message = "Carbon footprint must be greater than zero",
                value   = c.carbon_footprint_kg_co2_per_kwh,
            ))

        # Temperature range must be logical
        if s.operating_temp_min_c >= s.operating_temp_max_c:
            findings.append(ValidationFinding(
                level   = "ERROR",
                field   = "operating_temp_range",
                message = "Min operating temperature must be less than max",
                value   = f"min:{s.operating_temp_min_c}  max:{s.operating_temp_max_c}",
            ))

        # ── Category 2: Regulatory compliance ─────────────────────────

        # Recycled content cannot exceed 100%
        for field_name, value in [
            ("recycled_cobalt_pct",  c.recycled_cobalt_pct),
            ("recycled_lithium_pct", c.recycled_lithium_pct),
            ("recycled_nickel_pct",  c.recycled_nickel_pct),
        ]:
            if not (0.0 <= value <= 100.0):
                findings.append(ValidationFinding(
                    level   = "ERROR",
                    field   = field_name,
                    message = "Recycled content percentage must be 0–100%",
                    value   = value,
                ))

        # ── Category 3: Internal consistency ──────────────────────────

        # Energy content should approximately equal Ah × V / 1000
        # Allow 10% tolerance for different measurement conditions
        expected_kwh = (s.nominal_capacity_ah * s.nominal_voltage_v) / 1000
        actual_kwh   = s.energy_content_kwh
        if expected_kwh > 0:
            deviation = abs(actual_kwh - expected_kwh) / expected_kwh
            if deviation > 0.10:
                findings.append(ValidationFinding(
                    level   = "WARNING",
                    field   = "energy_content_kwh",
                    message = (
                        f"Energy content {actual_kwh} kWh deviates "
                        f"{deviation*100:.1f}% from calculated "
                        f"{expected_kwh:.1f} kWh (Ah × V / 1000) — "
                        f"verify pack specification"
                    ),
                    value   = actual_kwh,
                ))

        # SoH EOL threshold must be below second-life threshold
        if s.soh_eol_threshold_pct >= s.soh_second_life_threshold_pct:
            findings.append(ValidationFinding(
                level   = "ERROR",
                field   = "soh_thresholds",
                message = (
                    "EOL threshold must be below second-life threshold — "
                    "a battery cannot reach end-of-life before second-life"
                ),
                value   = (
                    f"eol:{s.soh_eol_threshold_pct}%  "
                    f"second_life:{s.soh_second_life_threshold_pct}%"
                ),
            ))

        # Manufacturing date cannot be in the future
        from datetime import date as _date
        if s.manufacturing_date > _date.today():
            findings.append(ValidationFinding(
                level   = "ERROR",
                field   = "manufacturing_date",
                message = "Manufacturing date cannot be in the future",
                value   = s.manufacturing_date.isoformat(),
            ))

        # Passport ID must not be empty
        if not passport.passport_id or len(passport.passport_id.strip()) == 0:
            findings.append(ValidationFinding(
                level   = "ERROR",
                field   = "passport_id",
                message = "Passport ID cannot be empty",
                value   = passport.passport_id,
            ))

        # ── Build and return report ────────────────────────────────────
        error_count = sum(1 for f in findings if f.level == "ERROR")
        return ValidationReport(
            passed   = error_count == 0,
            findings = findings,
        )


# ── Run validation on the current passport ─────────────────────────────

validator         = DataValidationAgent()
validation_report = validator.validate(passport)

# Always print the full report first
validation_report.print_report()

# Print individual findings for debugging
if validation_report.findings:
    print()
    print("  Detailed findings:")
    for f in validation_report.findings:
        print(f)
    print()

# Block execution if validation fails
if not validation_report.passed:
    raise ValueError(
        f"❌ Passport failed validation with "
        f"{validation_report.error_count} error(s). "
        f"Fix all ERRORs before running the arbitration pipeline."
    )

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  DATA VALIDATION REPORT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Result   : ✅ PASSED
  Errors   : 0
  Warnings : 0

  All checks passed — no issues found.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


In [25]:
# ══════════════════════════════════════════════════════════════════════
# ALA CELL 3 — DynamicCarbonTracker
# ══════════════════════════════════════════════════════════════════════
# Tracks the battery's full lifetime carbon footprint:
#   baseline  = manufacturing carbon declared in the JTC 24 layer
#   operational = carbon accumulated charge-by-charge from the grid
#   lifetime  = baseline + operational
#
# Design principle: the tracker is an append-only log.
# Nothing is ever overwritten — the full audit trail is preserved.
# ══════════════════════════════════════════════════════════════════════

@dataclass
class ChargeSession:
    """
    A single charging event.

    To connect a real BMS or telematics feed, populate these fields
    from your data source. The only two that drive carbon calculation
    are grid_region_code and kwh_delivered — everything else is
    metadata for the audit trail.
    """
    session_id:         str
    grid_region_code:   str    # Must match a key in GRID
    kwh_delivered:      float  # Actual energy delivered this session
    timestamp_utc:      str = field(
        default_factory=lambda: datetime.now(timezone.utc).isoformat()
    )

    def __post_init__(self):
        if self.grid_region_code not in GRID:
            raise ValueError(
                f"Unknown grid region '{self.grid_region_code}'. "
                f"Valid codes: {list(GRID.keys())}"
            )
        if self.kwh_delivered <= 0:
            raise ValueError("kwh_delivered must be greater than zero.")

    @property
    def kg_co2(self) -> float:
        """Carbon emitted this session in kg CO2-equivalent."""
        intensity = GRID[self.grid_region_code].intensity_g_per_kwh
        return round((intensity * self.kwh_delivered) / 1000, 4)


class DynamicCarbonTracker:
    """
    Tracks full lifetime carbon of a battery pack.

    Combines the static manufacturing declaration from the JTC 24
    circularity layer with operational emissions from real charging
    sessions. Every session is logged — nothing is ever lost.

    The key output — carbon_saving_vs_new_pct — answers the question
    the arbitrator needs: is keeping this battery in service genuinely
    greener than manufacturing a fresh replacement pack?
    """

    def __init__(self, passport: BatteryPassport):
        self.passport  = passport
        self.sessions: list[ChargeSession] = []

        # Manufacturing baseline from JTC 24
        cf  = passport.circularity.carbon_footprint_kg_co2_per_kwh
        kwh = passport.strategic.energy_content_kwh
        self.baseline_kg_co2: float = round(cf * kwh, 4)

    # ── Recording sessions ────────────────────────────────────────────

    def record_charge_session(self, session: ChargeSession) -> float:
        """
        Record one charging event.
        Returns kg CO2 emitted this session for immediate feedback.
        """
        self.sessions.append(session)
        return session.kg_co2

    def record_bulk_sessions(self, sessions: list[ChargeSession]) -> None:
        """Load a list of historical sessions in one call."""
        for s in sessions:
            self.record_charge_session(s)

    # ── Computed properties ───────────────────────────────────────────

    @property
    def operational_kg_co2(self) -> float:
        """Total carbon from all recorded charging sessions."""
        return round(sum(s.kg_co2 for s in self.sessions), 4)

    @property
    def lifetime_kg_co2(self) -> float:
        """Manufacturing baseline + all operational charging carbon."""
        return round(self.baseline_kg_co2 + self.operational_kg_co2, 4)

    @property
    def total_kwh_delivered(self) -> float:
        """Cumulative energy delivered across all sessions."""
        return round(sum(s.kwh_delivered for s in self.sessions), 4)

    @property
    def lifetime_intensity_g_per_kwh(self) -> Optional[float]:
        """
        Effective carbon intensity across the battery's operational life.
        Normalises total lifetime carbon by total useful energy delivered.
        This is the number that matters for ESG reporting — it makes
        batteries charged on clean grids directly comparable to those
        on heavy grids regardless of total energy delivered.
        """
        if self.total_kwh_delivered == 0:
            return None
        return round(
            (self.operational_kg_co2 * 1000) / self.total_kwh_delivered, 2
        )

    @property
    def carbon_saving_vs_new_pct(self) -> Optional[float]:
        """
        Carbon saving from keeping this battery in service versus
        manufacturing a brand-new replacement pack.

        Positive = second-life reuse is greener than a new battery.
        Negative = this battery has already accumulated so much
                   operational carbon that a new pack would be cleaner.

        This feeds directly into the CircularScore in ALA Cell 6.
        The higher this number, the stronger the case for second life.
        """
        if self.total_kwh_delivered == 0:
            return None
        new_pack_cf_g = (
            self.passport.circularity.carbon_footprint_kg_co2_per_kwh * 1000
        )
        amortised_g = (
            (self.lifetime_kg_co2 / self.total_kwh_delivered) * 1000
        )
        saving = ((new_pack_cf_g - amortised_g) / new_pack_cf_g) * 100
        return round(saving, 2)

    # ── Grid breakdown ────────────────────────────────────────────────

    def carbon_by_region(self) -> dict[str, float]:
        """
        Total kg CO2 attributed to each grid region.
        Useful for supply chain carbon reporting — shows exactly
        where in the world the operational carbon was emitted.
        """
        breakdown: dict[str, float] = {}
        for s in self.sessions:
            breakdown[s.grid_region_code] = round(
                breakdown.get(s.grid_region_code, 0.0) + s.kg_co2, 4
            )
        return dict(sorted(breakdown.items(), key=lambda x: x[1], reverse=True))

    # ── Export ────────────────────────────────────────────────────────

    def to_dict(self) -> dict:
        """
        Canonical dictionary — ready for SHA-256 hashing and
        inclusion in the ArbitrationReport in ALA Cell 9.
        """
        return {
            "passport_id":                  self.passport.passport_id,
            "baseline_kg_co2":              self.baseline_kg_co2,
            "operational_kg_co2":           self.operational_kg_co2,
            "lifetime_kg_co2":              self.lifetime_kg_co2,
            "total_kwh_delivered":          self.total_kwh_delivered,
            "lifetime_intensity_g_per_kwh": self.lifetime_intensity_g_per_kwh,
            "carbon_saving_vs_new_pct":     self.carbon_saving_vs_new_pct,
            "carbon_by_region":             self.carbon_by_region(),
            "session_count":                len(self.sessions),
            "_module":                      "DynamicCarbonTracker",
        }

    # ── Summary ───────────────────────────────────────────────────────

    def print_summary(self) -> None:
        saving = self.carbon_saving_vs_new_pct
        arrow  = "↓ cleaner" if (saving or 0) > 0 else "↑ heavier"

        print("━" * 58)
        print("  DYNAMIC CARBON TRACKER")
        print("━" * 58)
        print(f"  Manufacturing baseline : {self.baseline_kg_co2:>10.2f} kg CO2")
        print(f"  Operational (charging) : {self.operational_kg_co2:>10.2f} kg CO2")
        print(f"  Lifetime total         : {self.lifetime_kg_co2:>10.2f} kg CO2")
        print(f"  Total kWh delivered    : {self.total_kwh_delivered:>10.2f} kWh")
        print(f"  Charging sessions      : {len(self.sessions):>10}")
        if self.lifetime_intensity_g_per_kwh:
            print(f"  Lifetime intensity     : "
                  f"{self.lifetime_intensity_g_per_kwh:>10.1f} gCO2/kWh")
        if saving is not None:
            print(f"  vs new pack            : "
                  f"{abs(saving):>10.1f}%  {arrow} than manufacturing new")
        print()
        print("  Carbon by grid region:")
        for code, kg in self.carbon_by_region().items():
            name = GRID[code].name
            print(f"    {code}  {name:<14} {kg:>8.3f} kg CO2")
        print("━" * 58)


print("✅ DynamicCarbonTracker defined")
print("   ChargeSession  — one charging event")
print("   DynamicCarbonTracker — full lifetime carbon log")

✅ DynamicCarbonTracker defined
   ChargeSession  — one charging event
   DynamicCarbonTracker — full lifetime carbon log


In [26]:
# ══════════════════════════════════════════════════════════════════════
# ALA CELL 4 — Build the tracker and load a charging history
# ══════════════════════════════════════════════════════════════════════
# This cell is your data entry point. In production, replace the
# simulated_sessions list with records pulled from your BMS,
# telematics platform, or smart charging network API.
#
# Every session needs three things:
#   session_id        — any unique string (your BMS event ID)
#   grid_region_code  — a key from GRID (ALA Cell 2)
#   kwh_delivered     — actual energy delivered that session
# ══════════════════════════════════════════════════════════════════════

# Attach tracker to the passport already in memory
tracker = DynamicCarbonTracker(passport)

# ── Simulated charging history ─────────────────────────────────────────
# Scenario: NMC 75 kWh EV pack, two years of use across three countries.
#
# Year 1 — Owner commutes daily in Germany (coal-gas mix grid)
# Mid-year — Summer road trip through France and Norway (clean grids)
# Year 2 — Owner relocates to Poland (coal-heavy grid)
#
# This scenario is deliberately varied so you can see how grid mix
# affects the lifetime carbon figure and the carbon saving argument.

simulated_sessions = [

    # ── Year 1: Germany daily commute ─────────────────────────────────
    ChargeSession("S001", "DE", 62.0),
    ChargeSession("S002", "DE", 58.5),
    ChargeSession("S003", "DE", 71.0),
    ChargeSession("S004", "DE", 63.5),
    ChargeSession("S005", "DE", 59.0),
    ChargeSession("S006", "DE", 68.0),
    ChargeSession("S007", "DE", 64.5),
    ChargeSession("S008", "DE", 60.0),

    # ── Summer trip: France then Norway (much cleaner grids) ───────────
    ChargeSession("S009", "FR", 70.0),
    ChargeSession("S010", "FR", 72.5),
    ChargeSession("S011", "NO", 74.0),
    ChargeSession("S012", "NO", 69.5),
    ChargeSession("S013", "NO", 71.0),
    ChargeSession("S014", "FR", 68.0),

    # ── Year 2: Poland relocation (heaviest grid in registry) ──────────
    ChargeSession("S015", "PL", 61.0),
    ChargeSession("S016", "PL", 64.0),
    ChargeSession("S017", "PL", 60.5),
    ChargeSession("S018", "PL", 63.5),
    ChargeSession("S019", "PL", 65.0),
    ChargeSession("S020", "PL", 62.0),
]

# Load all sessions into the tracker
tracker.record_bulk_sessions(simulated_sessions)

# ── Full summary ───────────────────────────────────────────────────────
tracker.print_summary()

# ── Session-level detail ───────────────────────────────────────────────
print()
print(f"  {'Session':<8} {'Grid':<6} {'Country':<14} {'kWh':>7}  {'kg CO2':>8}")
print(f"  {'─'*52}")
running_total = 0.0
for s in tracker.sessions:
    running_total += s.kg_co2
    name = GRID[s.grid_region_code].name
    print(f"  {s.session_id:<8} {s.grid_region_code:<6} {name:<14} "
          f"{s.kwh_delivered:>7.1f}  {s.kg_co2:>8.3f}")
print(f"  {'─'*52}")
print(f"  {'TOTAL':<28} "
      f"{tracker.total_kwh_delivered:>7.1f}  "
      f"{tracker.operational_kg_co2:>8.3f}")

print()
print(f"  Key number for arbitration:")
print(f"  carbon_saving_vs_new_pct = {tracker.carbon_saving_vs_new_pct}%")
print(f"  → A positive number means second-life reuse is greener")
print(f"    than manufacturing a fresh replacement NMC pack.")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  DYNAMIC CARBON TRACKER
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Manufacturing baseline :    4875.00 kg CO2
  Operational (charging) :     371.89 kg CO2
  Lifetime total         :    5246.89 kg CO2
  Total kWh delivered    :    1307.50 kWh
  Charging sessions      :         20
  Lifetime intensity     :      284.4 gCO2/kWh
  vs new pack            :       93.8%  ↓ cleaner than manufacturing new

  Carbon by grid region:
    PL  Poland          205.296 kg CO2
    DE  Germany         145.872 kg CO2
    NO  Norway           16.087 kg CO2
    FR  France            4.631 kg CO2
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  Session  Grid   Country            kWh    kg CO2
  ────────────────────────────────────────────────────
  S001     DE     Germany           62.0    17.856
  S002     DE     Germany           58.5    16.848
  S003     DE     Germany           71.0    20.448
  S004     DE     Ger

In [27]:
# ══════════════════════════════════════════════════════════════════════
# ALA CELL 5 — Bid Dataclasses
# ══════════════════════════════════════════════════════════════════════
# Two market participants, two completely different valuation logics.
#
# RecyclerBid   — values the battery as a bag of raw materials.
#                 Price driven by commodity markets (Li, Co, Ni).
#
# SecondLifeBid — values the battery as an energy storage asset.
#                 Price driven by remaining kWh capacity.
#
# Both are fully parameterised — swap any float for a live API value.
#
# Live price feed examples:
#   Li spot price : London Metal Exchange API
#   Co spot price : Fastmarkets API
#   kWh ESS price : IRENA battery storage cost database
# ══════════════════════════════════════════════════════════════════════


# ── Material composition assumptions (NMC chemistry) ──────────────────
# These represent the kg of each critical material per kWh of capacity.
# Source: BloombergNEF Battery Price Survey 2023.
# Parameterise these for different chemistries (LFP has no Co).

MATERIAL_KG_PER_KWH = {
    "Co": 0.15,   # kg cobalt per kWh — NMC811 benchmark
    "Li": 0.11,   # kg lithium per kWh
    "Ni": 0.65,   # kg nickel per kWh
    "Mn": 0.12,   # kg manganese per kWh
}


@dataclass
class RecyclerBid:
    """
    A bid from a materials recovery operator.

    The recycler values the battery purely as a source of raw materials.
    Their offer is built from commodity spot prices multiplied by
    expected material yields after the hydrometallurgical process.

    To connect live commodity prices:
        bid = RecyclerBid(
            bidder_id     = "recycler_001",
            li_price_per_kg = lme_api.get_price("Li"),
            co_price_per_kg = fastmarkets_api.get_price("Co"),
            ni_price_per_kg = lme_api.get_price("Ni"),
            recovery_efficiency_pct = 0.92,
        )
    """
    # ── Identity ───────────────────────────────────────────────────────
    bidder_id:              str         # Recycler company identifier
    bidder_name:            str         # Human-readable name

    # ── Commodity prices (parameterise these for live feeds) ───────────
    li_price_per_kg:        float       # USD/kg lithium carbonate equiv.
    co_price_per_kg:        float       # USD/kg cobalt sulphate equiv.
    ni_price_per_kg:        float       # USD/kg nickel sulphate equiv.
    mn_price_per_kg:        float       # USD/kg manganese sulphate equiv.

    # ── Process parameters ─────────────────────────────────────────────
    recovery_efficiency_pct: float      # Fraction of material recovered
                                        # after hydrometallurgical process
                                        # (0.0 – 1.0, typically 0.85–0.95)

    # ── Metadata ───────────────────────────────────────────────────────
    bid_timestamp_utc: str = field(
        default_factory=lambda: datetime.now(timezone.utc).isoformat()
    )
    notes: str = ""

    def __post_init__(self):
        assert 0.0 < self.recovery_efficiency_pct <= 1.0, \
            "recovery_efficiency_pct must be between 0 and 1"
        for price_attr in ["li_price_per_kg", "co_price_per_kg",
                           "ni_price_per_kg", "mn_price_per_kg"]:
            assert getattr(self, price_attr) >= 0, \
                f"{price_attr} cannot be negative"

    def material_value(self, energy_kwh: float) -> float:
        """
        Total recoverable material value for a pack of given energy.

        Formula:
          material_value = Σ (kg_per_kwh × energy_kwh
                           × price_per_kg × recovery_efficiency)

        This is the gross material value before processing costs.
        Processing costs are absorbed into the recycler's margin —
        not modelled here to keep the interface clean.
        """
        li_val = (MATERIAL_KG_PER_KWH["Li"] * energy_kwh
                  * self.li_price_per_kg * self.recovery_efficiency_pct)
        co_val = (MATERIAL_KG_PER_KWH["Co"] * energy_kwh
                  * self.co_price_per_kg * self.recovery_efficiency_pct)
        ni_val = (MATERIAL_KG_PER_KWH["Ni"] * energy_kwh
                  * self.ni_price_per_kg * self.recovery_efficiency_pct)
        mn_val = (MATERIAL_KG_PER_KWH["Mn"] * energy_kwh
                  * self.mn_price_per_kg * self.recovery_efficiency_pct)
        return round(li_val + co_val + ni_val + mn_val, 2)

    def to_dict(self) -> dict:
        return {
            "type":                     "RecyclerBid",
            "bidder_id":                self.bidder_id,
            "bidder_name":              self.bidder_name,
            "li_price_per_kg":          self.li_price_per_kg,
            "co_price_per_kg":          self.co_price_per_kg,
            "ni_price_per_kg":          self.ni_price_per_kg,
            "mn_price_per_kg":          self.mn_price_per_kg,
            "recovery_efficiency_pct":  self.recovery_efficiency_pct,
            "bid_timestamp_utc":        self.bid_timestamp_utc,
            "notes":                    self.notes,
        }


@dataclass
class SecondLifeBid:
    """
    A bid from a stationary energy storage operator.

    The second-life operator values the battery as an energy asset.
    They do not care about chemistry or individual material prices —
    they care about how many usable kilowatt-hours they are buying
    and whether the pack is safe and reliable enough for their use case.

    March 2026 update — application-specific RTE thresholds:
      Different second-life applications have different efficiency
      requirements. A grid frequency response system charges and
      discharges many times per day — it needs high RTE because
      every cycle compounds the loss. A solar buffer charges once
      per day and can tolerate lower RTE.

      The min_rte_pct field encodes this requirement explicitly.
      If the battery's RTE is below the operator's minimum,
      the bid is withdrawn — just like SoH withdrawal.

    Application presets for convenience:
      SecondLifeBid.for_application("grid_frequency_response", ...)
      SecondLifeBid.for_application("solar_buffer_storage", ...)
      SecondLifeBid.for_application("backup_power", ...)
      SecondLifeBid.for_application("ev_charging_buffer", ...)

    To connect live ESS market prices:
        bid = SecondLifeBid(
            bidder_id          = "sl_operator_001",
            price_per_kwh_usd  = irena_api.get_ess_price("stationary"),
            min_soh_pct        = 75.0,
            min_rte_pct        = 85.0,
            target_application = "Grid frequency response",
        )
    """

    # ── Identity ───────────────────────────────────────────────────────
    bidder_id:          str
    bidder_name:        str

    # ── Pricing (parameterise for live ESS market feeds) ───────────────
    price_per_kwh_usd:  float       # USD per usable kWh of capacity
                                    # Typical range: $40–$120/kWh
                                    # for refurbished ESS applications

    # ── Technical requirements ─────────────────────────────────────────
    min_soh_pct:        float       # Minimum SoH they will accept
                                    # Below this → bid is withdrawn

    min_rte_pct:        float = 85.0  # Minimum round-trip efficiency (%)
                                      # Below this → bid is withdrawn
                                      # Varies by application — see presets

    max_c_rate:         float = 1.0   # Maximum C-rate their system needs
                                      # Low C-rate apps accept more degraded packs

    target_application: str = "Stationary Energy Storage"

    # ── Metadata ───────────────────────────────────────────────────────
    bid_timestamp_utc: str = field(
        default_factory=lambda: datetime.now(timezone.utc).isoformat()
    )
    notes: str = ""

    # ── Application presets ────────────────────────────────────────────
    # These reflect real-world operator requirements by application type.
    # Source: IRENA Battery Storage Report 2023, BloombergNEF ESS Survey.
    #
    #   grid_frequency_response  — charges/discharges many times per day
    #                              needs highest RTE — efficiency losses compound
    #
    #   solar_buffer_storage     — charges once per day, discharges overnight
    #                              tolerates lower RTE, lower SoH acceptable
    #
    #   backup_power             — rarely cycles, sits on standby
    #                              lowest RTE requirement — rarely discharges
    #
    #   ev_charging_buffer       — moderate cycling, peak shaving use case
    #                              mid-range requirements

    APPLICATION_PRESETS: ClassVar[dict] = {
        "grid_frequency_response": {
            "min_rte_pct": 87.0,
            "min_soh_pct": 78.0,
            "max_c_rate":  2.0,
            "description": "High-frequency grid balancing — highest efficiency required",
        },
        "solar_buffer_storage": {
            "min_rte_pct": 78.0,
            "min_soh_pct": 72.0,
            "max_c_rate":  0.5,
            "description": "Daily solar storage — moderate efficiency acceptable",
        },
        "backup_power": {
            "min_rte_pct": 75.0,
            "min_soh_pct": 70.0,
            "max_c_rate":  0.3,
            "description": "Standby backup — lowest cycling, most tolerant",
        },
        "ev_charging_buffer": {
            "min_rte_pct": 83.0,
            "min_soh_pct": 75.0,
            "max_c_rate":  1.5,
            "description": "EV charging peak shaving — moderate requirements",
        },
    }

    def __post_init__(self):
        assert self.price_per_kwh_usd >= 0, \
            "price_per_kwh_usd cannot be negative"
        assert 0 < self.min_soh_pct <= 100, \
            "min_soh_pct must be between 0 and 100"
        assert 0 < self.min_rte_pct <= 100, \
            "min_rte_pct must be between 0 and 100"

    @classmethod
    def for_application(
        cls,
        application:      str,
        bidder_id:        str,
        bidder_name:      str,
        price_per_kwh_usd: float,
        notes:            str = "",
    ) -> "SecondLifeBid":
        """
        Convenience constructor using application presets.

        Usage:
            bid = SecondLifeBid.for_application(
                application       = "solar_buffer_storage",
                bidder_id         = "sl_001",
                bidder_name       = "GridStore Europe BV",
                price_per_kwh_usd = 78.00,
            )

        This automatically sets min_rte_pct, min_soh_pct, and max_c_rate
        to the correct values for that application type.
        """
        if application not in cls.APPLICATION_PRESETS:
            valid = list(cls.APPLICATION_PRESETS.keys())
            raise ValueError(
                f"Unknown application '{application}'. "
                f"Valid options: {valid}"
            )
        preset = cls.APPLICATION_PRESETS[application]
        return cls(
            bidder_id         = bidder_id,
            bidder_name       = bidder_name,
            price_per_kwh_usd = price_per_kwh_usd,
            min_soh_pct       = preset["min_soh_pct"],
            min_rte_pct       = preset["min_rte_pct"],
            max_c_rate        = preset["max_c_rate"],
            target_application = application,
            notes             = notes or preset["description"],
        )

    def bid_value(
        self,
        remaining_kwh: float,
        soh_pct:       float,
        rte_pct:       float = 100.0,
    ) -> Optional[float]:
        """
        Total bid value for a pack with given remaining energy.

        Returns None if the pack's SoH OR RTE is below the bidder's
        minimum — meaning the bid is technically withdrawn for this pack.

        Both conditions must be met for the bid to stand:
          1. soh_pct >= min_soh_pct
          2. rte_pct >= min_rte_pct

        Formula:
          bid_value = remaining_kwh × price_per_kwh_usd
        """
        if soh_pct < self.min_soh_pct:
            return None     # Bid withdrawn — SoH below minimum
        if rte_pct < self.min_rte_pct:
            return None     # Bid withdrawn — RTE below application minimum
        return round(remaining_kwh * self.price_per_kwh_usd, 2)

    def withdrawal_reason(
        self,
        soh_pct: float,
        rte_pct: float = 100.0,
    ) -> Optional[str]:
        """
        Returns a human-readable reason if the bid is withdrawn.
        Returns None if the bid stands.
        Useful for the reasoning chain in RetirementVerdict.
        """
        if soh_pct < self.min_soh_pct:
            return (
                f"SoH {soh_pct}% is below operator minimum "
                f"{self.min_soh_pct}% for {self.target_application}"
            )
        if rte_pct < self.min_rte_pct:
            return (
                f"RTE {rte_pct}% is below operator minimum "
                f"{self.min_rte_pct}% for {self.target_application} — "
                f"efficiency losses make this application unviable"
            )
        return None

    def to_dict(self) -> dict:
        return {
            "type":               "SecondLifeBid",
            "bidder_id":          self.bidder_id,
            "bidder_name":        self.bidder_name,
            "price_per_kwh_usd":  self.price_per_kwh_usd,
            "min_soh_pct":        self.min_soh_pct,
            "min_rte_pct":        self.min_rte_pct,
            "max_c_rate":         self.max_c_rate,
            "target_application": self.target_application,
            "bid_timestamp_utc":  self.bid_timestamp_utc,
            "notes":              self.notes,
        }

    def print_requirements(self) -> None:
        """Print this operator's technical requirements clearly."""
        print(f"  {self.bidder_name} — {self.target_application}")
        print(f"    Price        : ${self.price_per_kwh_usd}/kWh")
        print(f"    Min SoH      : {self.min_soh_pct}%")
        print(f"    Min RTE      : {self.min_rte_pct}%")
        print(f"    Max C-rate   : {self.max_c_rate}C")
        if self.notes:
            print(f"    Notes        : {self.notes}")


# ── Preview: instantiate both bids and compute their values ───────────

# ── Preview: instantiate both bids and compute their values ───────────

# Recycler bid — using approximate 2024 commodity prices
recycler_bid = RecyclerBid(
    bidder_id               = "recycler_001",
    bidder_name             = "EuroRecycle GmbH",
    li_price_per_kg         = 13.00,
    co_price_per_kg         = 28.00,
    ni_price_per_kg         = 14.00,
    mn_price_per_kg         =  2.00,
    recovery_efficiency_pct = 0.92,
    notes                   = "Hydrometallurgical process, DIN EN ISO certified",
)

# Second-life bid — using application preset
second_life_bid = SecondLifeBid.for_application(
    application       = "solar_buffer_storage",
    bidder_id         = "sl_operator_001",
    bidder_name       = "GridStore Europe BV",
    price_per_kwh_usd = 78.00,
    notes             = "10-year offtake contract, ISO 14001 certified facility",
)

# Show requirements
second_life_bid.print_requirements()

# Compute values against our actual battery
energy_kwh    = passport.strategic.energy_content_kwh
soh_pct       = passport.circularity.current_soh_pct
remaining_kwh = passport.assess_second_life()["remaining_energy_kwh"]

recycler_value    = recycler_bid.material_value(energy_kwh)
second_life_value = second_life_bid.bid_value(remaining_kwh, soh_pct)

print()
print("━" * 58)
print("  BID SUMMARY")
print("━" * 58)
print(f"  Battery : {passport.strategic.model_designation}")
print(f"  Energy  : {energy_kwh} kWh nominal  |  {remaining_kwh} kWh remaining")
print(f"  SoH     : {soh_pct}%")
print()
print(f"  Recycler offer    : ${recycler_value:>8.2f}")
if second_life_value:
    print(f"  Second-life offer : ${second_life_value:>8.2f}")
    diff = second_life_value - recycler_value
    winner_on_price = "Second-Life" if diff > 0 else "Recycler"
    print(f"  Price difference  : ${abs(diff):>8.2f}  "
          f"({winner_on_price} higher by price alone)")
else:
    print(f"  Second-life offer : WITHDRAWN")
print()
print("  ⚠️  Price alone does not decide — the CircularScore in")
print("     ALA Cell 6 applies waste hierarchy weighting on top.")
print("━" * 58)

  GridStore Europe BV — solar_buffer_storage
    Price        : $78.0/kWh
    Min SoH      : 72.0%
    Min RTE      : 78.0%
    Max C-rate   : 0.5C
    Notes        : 10-year offtake contract, ISO 14001 certified facility

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  BID SUMMARY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Battery : EV-NMC-75kWh-Gen2
  Energy  : 75.0 kWh nominal  |  67.5 kWh remaining
  SoH     : 90.0%

  Recycler offer    : $ 1032.93
  Second-life offer : $ 5265.00
  Price difference  : $ 4232.07  (Second-Life higher by price alone)

  ⚠️  Price alone does not decide — the CircularScore in
     ALA Cell 6 applies waste hierarchy weighting on top.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


In [28]:
# ══════════════════════════════════════════════════════════════════════
# ALA CELL 6 — CircularScore
# ══════════════════════════════════════════════════════════════════════
# A weighted multi-criteria decision model that scores both bids
# on four dimensions simultaneously:
#
#   1. Financial value        weight: 35%
#   2. State of Health        weight: 30%
#   3. Carbon saving          weight: 20%
#   4. Waste hierarchy bonus  weight: 15%
#
# Weights sum to 1.0. Adjust them to reflect your organisation's
# priorities — more ESG focus → increase carbon weight,
# more commercial focus → increase financial weight.
#
# All individual scores are normalised to 0.0–1.0 before weighting
# so that no single criterion dominates purely through scale.
# ══════════════════════════════════════════════════════════════════════


@dataclass
class ScoringWeights:
    """
    Parameterised weights for the CircularScore model.

    All four weights must sum to exactly 1.0.
    Default values reflect the EU waste hierarchy priority order —
    reuse over recycling, with financial value as the primary
    commercial anchor.

    Customise for your use case:
      ESG-first organisation  → raise carbon_saving, lower financial
      Pure commercial         → raise financial, lower hierarchy_bonus
      Regulator / auditor     → raise hierarchy_bonus to maximum
    """
    financial:       float = 0.35
    soh:             float = 0.30
    carbon_saving:   float = 0.20
    hierarchy_bonus: float = 0.15

    def __post_init__(self):
        total = round(
            self.financial + self.soh +
            self.carbon_saving + self.hierarchy_bonus, 6
        )
        assert total == 1.0, (
            f"Weights must sum to 1.0 — current sum: {total}. "
            f"Adjust your weights and try again."
        )


@dataclass
class BidScore:
    """
    The scored result for a single bid.
    Holds both the raw inputs and the normalised component scores
    so the arbitrator can explain its reasoning transparently.
    """
    bid_type:               str     # "RecyclerBid" or "SecondLifeBid"
    bidder_name:            str
    raw_financial_value:    float   # USD — raw offer amount
    financial_score:        float   # 0.0–1.0 normalised
    soh_score:              float   # 0.0–1.0 normalised
    carbon_score:           float   # 0.0–1.0 normalised
    hierarchy_score:        float   # 0.0–1.0 (bonus for second life)
    weighted_total:         float   # final composite score

    def print_scorecard(self) -> None:
        print(f"  {self.bidder_name} [{self.bid_type}]")
        print(f"    Financial value   : ${self.raw_financial_value:>8.2f}"
              f"  (score: {self.financial_score:.3f})")
        print(f"    SoH signal        :           "
              f"  (score: {self.soh_score:.3f})")
        print(f"    Carbon saving     :           "
              f"  (score: {self.carbon_score:.3f})")
        print(f"    Hierarchy bonus   :           "
              f"  (score: {self.hierarchy_score:.3f})")
        print(f"    ── Weighted total  :           "
              f"   score: {self.weighted_total:.4f}")


class CircularScore:
    """
    Scores both bids and returns a BidScore for each.

    Scoring methodology:
      Each criterion is first normalised to 0.0–1.0 relative to the
      competing bid, then multiplied by its weight. The four weighted
      components are summed to produce a final score between 0.0–1.0.

      Normalisation ensures that a recycler offering $1,200 vs a
      second-life operator offering $950 produces a financial score
      difference of ~0.44 — not a raw dollar gap that would swamp
      all other criteria.

    The waste hierarchy bonus is not normalised against the competing
    bid — it is an absolute score applied only to second-life bids,
    reflecting the EU legal priority for reuse over recycling.
    """

    def __init__(
        self,
        passport:        BatteryPassport,
        tracker:         DynamicCarbonTracker,
        weights:         ScoringWeights = None,
        soh_sl_threshold: float = 75.0,
    ):
        self.passport          = passport
        self.tracker           = tracker
        self.weights           = weights or ScoringWeights()
        self.soh_sl_threshold  = soh_sl_threshold  # SoH override threshold

        # Cache the key values used across all scoring methods
        self.current_soh       = passport.circularity.current_soh_pct
        self.remaining_kwh     = passport.assess_second_life()["remaining_energy_kwh"]
        self.carbon_saving     = tracker.carbon_saving_vs_new_pct or 0.0

    # ── Internal normalisation helpers ────────────────────────────────

    @staticmethod
    def _normalise(value: float, low: float, high: float) -> float:
        """
        Map a value to 0.0–1.0 between a known low and high.
        Returns 0.5 if low == high (avoids division by zero).
        Clamps to [0.0, 1.0] if value falls outside the range.
        """
        if high == low:
            return 0.5
        return round(max(0.0, min(1.0, (value - low) / (high - low))), 4)

    # ── Individual scoring criteria ───────────────────────────────────

    def _score_financial(
        self,
        recycler_value: float,
        sl_value: float,
    ) -> tuple[float, float]:
        """
        Normalise both financial offers against each other.
        Higher offer → score closer to 1.0.
        Returns (recycler_score, sl_score).
        """
        low, high = min(recycler_value, sl_value), max(recycler_value, sl_value)
        r_score = self._normalise(recycler_value, low, high)
        s_score = self._normalise(sl_value,       low, high)
        return r_score, s_score

    def _score_soh(self) -> tuple[float, float]:
        """
        SoH signal score.

        For the second-life bid: higher SoH → higher score.
        Normalised between 0% and 100% SoH.

        For the recycler bid: SoH is irrelevant — they are recovering
        materials regardless. The recycler always receives 0.5 here,
        reflecting neutrality rather than advantage or disadvantage.
        """
        sl_score = self._normalise(self.current_soh, 0.0, 100.0)
        r_score  = 0.5   # Recycler is SoH-neutral
        return r_score, sl_score

    def _score_carbon(self) -> tuple[float, float]:
        """
        Carbon saving score.

        Reflects how much greener second-life reuse is versus
        manufacturing a new replacement pack.

        carbon_saving_vs_new_pct range: typically -20% to +60%.
        We normalise this to 0.0–1.0 across a -50 to +100 range.

        Positive saving → second life wins on carbon.
        Negative saving → recycling + new pack is actually cleaner
                          (rare but possible in coal-heavy grids).

        Recycler always scores 0.5 on carbon — carbon accounting
        for the recycling process itself is out of scope here.
        """
        sl_score = self._normalise(self.carbon_saving, -50.0, 100.0)
        r_score  = 0.5   # Recycler is carbon-neutral in this model
        return r_score, sl_score

    def _score_hierarchy(self) -> tuple[float, float]:
        """
        Waste hierarchy bonus.

        The EU Waste Framework Directive mandates: reuse before recycling.
        This criterion translates that legal priority into a scoring
        advantage for second life.

        If SoH is above the second-life threshold, second life receives
        the full hierarchy bonus (1.0). Below threshold, the bonus
        tapers — the pack is approaching recycling territory anyway.

        Recycler always scores 0.0 on this criterion — they represent
        the outer loop of the waste hierarchy, which is never preferred
        when an inner loop (reuse) is available.
        """
        if self.current_soh >= self.soh_sl_threshold:
            sl_score = 1.0   # Full bonus — pack is clearly viable for reuse
        else:
            # Tapered bonus — pack is below ideal but not disqualified
            sl_score = self._normalise(
                self.current_soh,
                self.soh_sl_threshold - 20,   # lower bound for taper
                self.soh_sl_threshold,
            )
        r_score = 0.0
        return r_score, sl_score

    # ── Main scoring method ───────────────────────────────────────────

    def score(
        self,
        recycler_bid:    RecyclerBid,
        second_life_bid: SecondLifeBid,
    ) -> tuple[BidScore, BidScore]:
        """
        Score both bids and return a BidScore for each.

        Returns (recycler_score, second_life_score).
        The arbitrator in ALA Cell 7 compares the weighted totals
        and applies the SoH hard override rule on top.
        """
        # Compute raw financial values for this pack
        r_value  = recycler_bid.material_value(
            self.passport.strategic.energy_content_kwh
        )
        sl_value_raw = second_life_bid.bid_value(
            self.remaining_kwh,
            self.current_soh,
        )

        # Handle withdrawn second-life bid
        sl_value = sl_value_raw if sl_value_raw is not None else 0.0
        bid_withdrawn = sl_value_raw is None

        # Score each criterion
        r_fin,  sl_fin  = self._score_financial(r_value, sl_value)
        r_soh,  sl_soh  = self._score_soh()
        r_carb, sl_carb = self._score_carbon()
        r_hier, sl_hier = self._score_hierarchy()

        w = self.weights

        # Compute weighted totals
        r_total = round(
            r_fin  * w.financial  +
            r_soh  * w.soh        +
            r_carb * w.carbon_saving +
            r_hier * w.hierarchy_bonus,
            4
        )
        sl_total = round(
            sl_fin  * w.financial  +
            sl_soh  * w.soh        +
            sl_carb * w.carbon_saving +
            sl_hier * w.hierarchy_bonus,
            4
        ) if not bid_withdrawn else 0.0

        recycler_score = BidScore(
            bid_type            = "RecyclerBid",
            bidder_name         = recycler_bid.bidder_name,
            raw_financial_value = r_value,
            financial_score     = r_fin,
            soh_score           = r_soh,
            carbon_score        = r_carb,
            hierarchy_score     = r_hier,
            weighted_total      = r_total,
        )

        sl_score = BidScore(
            bid_type            = "SecondLifeBid",
            bidder_name         = second_life_bid.bidder_name,
            raw_financial_value = sl_value,
            financial_score     = sl_fin,
            soh_score           = sl_soh,
            carbon_score        = sl_carb,
            hierarchy_score     = sl_hier,
            weighted_total      = sl_total,
        )

        return recycler_score, sl_score


# ── Preview: run the score and print both scorecards ──────────────────

scorer = CircularScore(passport, tracker)
r_score, sl_score = scorer.score(recycler_bid, second_life_bid)

print("━" * 58)
print("  CIRCULAR SCORE — BOTH BIDS")
print("━" * 58)
print()
print("  Scoring weights:")
w = scorer.weights
print(f"    Financial      : {w.financial*100:.0f}%")
print(f"    State of Health: {w.soh*100:.0f}%")
print(f"    Carbon saving  : {w.carbon_saving*100:.0f}%")
print(f"    Hierarchy bonus: {w.hierarchy_bonus*100:.0f}%")
print()
print("  ── Recycler ─────────────────────────────────────")
r_score.print_scorecard()
print()
print("  ── Second-Life ──────────────────────────────────")
sl_score.print_scorecard()
print()
leader = "Second-Life" if sl_score.weighted_total > r_score.weighted_total \
         else "Recycler"
margin = abs(sl_score.weighted_total - r_score.weighted_total)
print(f"  Score leader  : {leader}")
print(f"  Score margin  : {margin:.4f}")
print()
print("  ⚠️  This is the pre-arbitration score.")
print("     The hard SoH override rule is applied in ALA Cell 7.")
print("━" * 58)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  CIRCULAR SCORE — BOTH BIDS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  Scoring weights:
    Financial      : 35%
    State of Health: 30%
    Carbon saving  : 20%
    Hierarchy bonus: 15%

  ── Recycler ─────────────────────────────────────
  EuroRecycle GmbH [RecyclerBid]
    Financial value   : $ 1032.93  (score: 0.000)
    SoH signal        :             (score: 0.500)
    Carbon saving     :             (score: 0.500)
    Hierarchy bonus   :             (score: 0.000)
    ── Weighted total  :              score: 0.2500

  ── Second-Life ──────────────────────────────────
  GridStore Europe BV [SecondLifeBid]
    Financial value   : $ 5265.00  (score: 1.000)
    SoH signal        :             (score: 0.900)
    Carbon saving     :             (score: 0.959)
    Hierarchy bonus   :             (score: 1.000)
    ── Weighted total  :              score: 0.9618

  Score leader  : Second-Life
  Score margin 

In [29]:
# ══════════════════════════════════════════════════════════════════════
# ALA CELL 7 — LifeCycleArbitrator
# ══════════════════════════════════════════════════════════════════════
# The decision engine. Takes both scored bids and produces a
# final, explainable, legally-grounded retirement decision.
#
# Decision logic has two layers:
#
#   Layer 1 — Hard rules (applied first, non-negotiable):
#     • If second-life bid is withdrawn → recycle, no scoring needed
#     • If SoH ≥ soh_sl_threshold AND second life scores within
#       the price_override_tolerance → second life wins regardless
#       of price. This enforces the EU waste hierarchy mandate.
#
#   Layer 2 — Circular Score (applied when no hard rule fires):
#     • The bid with the higher weighted_total wins.
#     • Full reasoning is attached to the decision.
#
# Confidence Score (new):
#     • HIGH   → margin > 0.20  — clear decision, no review needed
#     • MEDIUM → margin 0.10–0.20 — automated but flag for audit
#     • LOW    → margin < 0.10  — borderline, recommend human review
#
# Every decision is fully explainable — the arbitrator never
# produces a verdict without a documented reason.
# ══════════════════════════════════════════════════════════════════════


class ArbiterDecision(Enum):
    """Possible outcomes of negotiate_retirement()."""
    SECOND_LIFE_WINS      = "Second-Life Provider awarded contract"
    RECYCLER_WINS         = "Recycler awarded contract"
    SECOND_LIFE_OVERRIDE  = "Second-Life wins on waste hierarchy override"
    NO_VALID_BID          = "No valid bid — manual review required"


@dataclass
class RetirementVerdict:
    """
    The complete output of a negotiate_retirement() call.

    Contains:
      - The decision and the winning bidder
      - Full scores for both bids
      - The reasoning chain that produced the decision
      - The key battery metrics at the time of arbitration
      - A confidence score showing how certain the decision is
      - A timestamp for the audit trail

    Designed to be exported directly to the ArbitrationReport
    in ALA Cell 9 and incorporated into the passport JSON.
    """
    decision:           ArbiterDecision
    winner_name:        str
    winner_bid_type:    str
    recycler_score:     BidScore
    sl_score:           BidScore
    reasoning:          list[str]
    soh_at_decision:    float
    remaining_kwh:      float
    carbon_saving_pct:  Optional[float]
    timestamp_utc:      str = field(
        default_factory=lambda: datetime.now(timezone.utc).isoformat()
    )

    # ── Confidence score ───────────────────────────────────────────────
    # Measures how certain the arbitrator is about its decision.
    # Based on the margin between the two weighted scores.
    # HIGH   → margin > 0.20  — clear decision, no review needed
    # MEDIUM → margin 0.10–0.20 — automated but flag for audit
    # LOW    → margin < 0.10  — borderline, recommend human review

    @property
    def confidence_margin(self) -> float:
        """Raw score margin between winner and runner-up."""
        return round(abs(
            self.sl_score.weighted_total - self.recycler_score.weighted_total
        ), 4)

    @property
    def confidence(self) -> str:
        """Human-readable confidence level based on score margin."""
        margin = self.confidence_margin
        if margin > 0.20:
            return "HIGH"
        elif margin > 0.10:
            return "MEDIUM"
        else:
            return "LOW"

    @property
    def confidence_icon(self) -> str:
        return {"HIGH": "🟢", "MEDIUM": "🟡", "LOW": "🔴"}[self.confidence]

    @property
    def requires_human_review(self) -> bool:
        """True when confidence is LOW — system recommends human oversight."""
        return self.confidence == "LOW"

    def print_verdict(self) -> None:
        icon = {
            ArbiterDecision.SECOND_LIFE_WINS:     "♻️  SECOND LIFE",
            ArbiterDecision.RECYCLER_WINS:        "⛏️  RECYCLE",
            ArbiterDecision.SECOND_LIFE_OVERRIDE: "⚖️  SECOND LIFE (override)",
            ArbiterDecision.NO_VALID_BID:         "⚠️  MANUAL REVIEW",
        }[self.decision]

        print("━" * 62)
        print(f"  ARBITRATION VERDICT")
        print("━" * 62)
        print(f"  Decision    : {icon}")
        print(f"  Awarded to  : {self.winner_name}")
        print(f"  Bid type    : {self.winner_bid_type}")
        print(f"  SoH         : {self.soh_at_decision}%")
        print(f"  Remaining   : {self.remaining_kwh} kWh")
        if self.carbon_saving_pct is not None:
            print(f"  Carbon save : {self.carbon_saving_pct}% vs new pack")
        print()
        print(f"  Confidence  : {self.confidence_icon}  {self.confidence}"
              f"  (margin: {self.confidence_margin:.4f})")
        if self.requires_human_review:
            print(f"  ⚠️  LOW CONFIDENCE — human review recommended before executing")
        print()
        print("  Scores:")
        print(f"    {self.recycler_score.bidder_name:<28} "
              f"{self.recycler_score.weighted_total:.4f}")
        print(f"    {self.sl_score.bidder_name:<28} "
              f"{self.sl_score.weighted_total:.4f}")
        print()
        print("  Reasoning chain:")
        for i, step in enumerate(self.reasoning, 1):
            print(f"    {i}. {step}")
        print(f"  Timestamp   : {self.timestamp_utc}")
        print("━" * 62)

    def to_dict(self) -> dict:
        return {
            "decision":              self.decision.value,
            "winner_name":           self.winner_name,
            "winner_bid_type":       self.winner_bid_type,
            "soh_at_decision":       self.soh_at_decision,
            "remaining_kwh":         self.remaining_kwh,
            "carbon_saving_pct":     self.carbon_saving_pct,
            "timestamp_utc":         self.timestamp_utc,
            "confidence":            self.confidence,
            "confidence_margin":     self.confidence_margin,
            "requires_human_review": self.requires_human_review,
            "recycler_score": {
                "bidder":          self.recycler_score.bidder_name,
                "financial":       self.recycler_score.financial_score,
                "soh":             self.recycler_score.soh_score,
                "carbon":          self.recycler_score.carbon_score,
                "hierarchy":       self.recycler_score.hierarchy_score,
                "weighted_total":  self.recycler_score.weighted_total,
            },
            "sl_score": {
                "bidder":          self.sl_score.bidder_name,
                "financial":       self.sl_score.financial_score,
                "soh":             self.sl_score.soh_score,
                "carbon":          self.sl_score.carbon_score,
                "hierarchy":       self.sl_score.hierarchy_score,
                "weighted_total":  self.sl_score.weighted_total,
            },
            "reasoning": self.reasoning,
        }


class LifeCycleArbitrator:
    """
    The Autonomous Life-Cycle Arbitrator.

    Accepts competing bids from a recycler and a second-life operator,
    scores them through the CircularScore model, applies hard override
    rules grounded in the EU waste hierarchy, and produces a fully
    explainable RetirementVerdict with a confidence score.

    Key parameter — price_override_tolerance:
      How much more the recycler can offer (as a fraction) before
      the SoH-based second-life override is defeated by price.
      Default 0.15 means: second life wins on override even if
      recycler bids up to 15% more, as long as SoH ≥ threshold.
      Set to 0.0 for a strict override regardless of price gap.
      Set to 0.30 for a more commercially-flexible override.
    """

    def __init__(
        self,
        passport:                 BatteryPassport,
        tracker:                  DynamicCarbonTracker,
        weights:                  ScoringWeights = None,
        soh_sl_threshold:         float = 75.0,
        price_override_tolerance: float = 0.15,
    ):
        self.passport                 = passport
        self.tracker                  = tracker
        self.scorer                   = CircularScore(
            passport, tracker, weights, soh_sl_threshold
        )
        self.soh_sl_threshold         = soh_sl_threshold
        self.price_override_tolerance = price_override_tolerance

    def negotiate_retirement(
        self,
        recycler_bid:    RecyclerBid,
        second_life_bid: SecondLifeBid,
    ) -> RetirementVerdict:
        """
        The core arbitration method.

        Call this once per battery reaching end of primary use.
        Returns a RetirementVerdict with full decision, scores,
        confidence level, and reasoning — ready for export and audit.
        """
        reasoning: list[str] = []
        soh        = self.passport.circularity.current_soh_pct
        remaining  = self.passport.assess_second_life()["remaining_energy_kwh"]
        carbon_sav = self.tracker.carbon_saving_vs_new_pct

        # ── Score both bids ────────────────────────────────────────────
        r_score, sl_score = self.scorer.score(recycler_bid, second_life_bid)

        reasoning.append(
            f"Scored both bids — Recycler: {r_score.weighted_total:.4f} | "
            f"Second-Life: {sl_score.weighted_total:.4f}"
        )

        # ══ LAYER 1: HARD RULES ════════════════════════════════════════

        # Rule 1a — Second-life bid withdrawn (SoH below operator minimum)
        if sl_score.raw_financial_value == 0.0 and sl_score.weighted_total == 0.0:
            reasoning.append(
                f"Second-life bid withdrawn — SoH {soh}% is below "
                f"operator minimum {second_life_bid.min_soh_pct}%."
            )
            reasoning.append(
                "Hard rule 1a fired: no valid second-life offer. "
                "Routing to recycler by default."
            )
            return RetirementVerdict(
                decision          = ArbiterDecision.RECYCLER_WINS,
                winner_name       = recycler_bid.bidder_name,
                winner_bid_type   = "RecyclerBid",
                recycler_score    = r_score,
                sl_score          = sl_score,
                reasoning         = reasoning,
                soh_at_decision   = soh,
                remaining_kwh     = remaining,
                carbon_saving_pct = carbon_sav,
            )

        # Rule 1b — SoH override: second life wins even if recycler
        # bids higher, provided the price gap is within tolerance.
        # Grounded in EU Waste Framework Directive — reuse before recycling.
        if soh >= self.soh_sl_threshold:
            r_val  = r_score.raw_financial_value
            sl_val = sl_score.raw_financial_value
            price_gap_fraction = (r_val - sl_val) / r_val if r_val > 0 else 0.0

            reasoning.append(
                f"SoH {soh}% ≥ threshold {self.soh_sl_threshold}% — "
                f"second-life override rule is active."
            )

            if price_gap_fraction <= self.price_override_tolerance:
                reasoning.append(
                    f"Price gap {price_gap_fraction*100:.1f}% is within "
                    f"override tolerance {self.price_override_tolerance*100:.0f}%. "
                    f"EU waste hierarchy mandate: reuse preferred over recycling."
                )
                reasoning.append(
                    "Hard rule 1b fired: second-life wins on waste "
                    "hierarchy override."
                )
                return RetirementVerdict(
                    decision          = ArbiterDecision.SECOND_LIFE_OVERRIDE,
                    winner_name       = second_life_bid.bidder_name,
                    winner_bid_type   = "SecondLifeBid",
                    recycler_score    = r_score,
                    sl_score          = sl_score,
                    reasoning         = reasoning,
                    soh_at_decision   = soh,
                    remaining_kwh     = remaining,
                    carbon_saving_pct = carbon_sav,
                )
            else:
                reasoning.append(
                    f"Price gap {price_gap_fraction*100:.1f}% exceeds "
                    f"override tolerance {self.price_override_tolerance*100:.0f}%. "
                    f"Override not triggered — proceeding to Circular Score."
                )

        # ══ LAYER 2: CIRCULAR SCORE ════════════════════════════════════

        reasoning.append(
            "No hard rule fired — decision by Circular Score."
        )

        if sl_score.weighted_total >= r_score.weighted_total:
            reasoning.append(
                f"Second-life Circular Score {sl_score.weighted_total:.4f} ≥ "
                f"recycler score {r_score.weighted_total:.4f}. "
                f"Second-life wins on composite score."
            )
            return RetirementVerdict(
                decision          = ArbiterDecision.SECOND_LIFE_WINS,
                winner_name       = second_life_bid.bidder_name,
                winner_bid_type   = "SecondLifeBid",
                recycler_score    = r_score,
                sl_score          = sl_score,
                reasoning         = reasoning,
                soh_at_decision   = soh,
                remaining_kwh     = remaining,
                carbon_saving_pct = carbon_sav,
            )
        else:
            reasoning.append(
                f"Recycler Circular Score {r_score.weighted_total:.4f} > "
                f"second-life score {sl_score.weighted_total:.4f}. "
                f"Recycler wins on composite score."
            )
            return RetirementVerdict(
                decision          = ArbiterDecision.RECYCLER_WINS,
                winner_name       = recycler_bid.bidder_name,
                winner_bid_type   = "RecyclerBid",
                recycler_score    = r_score,
                sl_score          = sl_score,
                reasoning         = reasoning,
                soh_at_decision   = soh,
                remaining_kwh     = remaining,
                carbon_saving_pct = carbon_sav,
            )


# ── Run the arbitration ────────────────────────────────────────────────

arbitrator = LifeCycleArbitrator(
    passport                 = passport,
    tracker                  = tracker,
    soh_sl_threshold         = 75.0,
    price_override_tolerance = 0.15,
)

verdict = arbitrator.negotiate_retirement(recycler_bid, second_life_bid)
verdict.print_verdict()

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ARBITRATION VERDICT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Decision    : ⚖️  SECOND LIFE (override)
  Awarded to  : GridStore Europe BV
  Bid type    : SecondLifeBid
  SoH         : 90.0%
  Remaining   : 67.5 kWh
  Carbon save : 93.83% vs new pack

  Confidence  : 🟢  HIGH  (margin: 0.7118)

  Scores:
    EuroRecycle GmbH             0.2500
    GridStore Europe BV          0.9618

  Reasoning chain:
    1. Scored both bids — Recycler: 0.2500 | Second-Life: 0.9618
    2. SoH 90.0% ≥ threshold 75.0% — second-life override rule is active.
    3. Price gap -409.7% is within override tolerance 15%. EU waste hierarchy mandate: reuse preferred over recycling.
    4. Hard rule 1b fired: second-life wins on waste hierarchy override.
  Timestamp   : 2026-04-15T11:51:00.982542+00:00
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


In [30]:
# ══════════════════════════════════════════════════════════════════════
# ALA CELL 8 — PrivacyPreservingCert
# ══════════════════════════════════════════════════════════════════════
# A hash-based commitment scheme that proves safety claims about a
# battery without exposing the raw sensor data that supports them.
#
# Mechanism:
#   1. Define a set of safety claims (human-readable assertions)
#   2. Each claim is salted with a unique random value
#   3. SHA-256(claim + salt) → commitment hash
#   4. The certificate publishes: claim text + commitment hash
#      but NOT the salt (kept private by the issuer)
#   5. Selective disclosure: issuer can later reveal (claim + salt)
#      for any individual claim — verifier recomputes and confirms
#
# What the receiver can verify:
#   ✅ The certificate is authentic (master hash check)
#   ✅ The issuer committed to specific safety claims
#   ✅ Individual claims can be selectively verified if salt revealed
#
# What the receiver cannot learn:
#   ❌ The actual sensor readings behind each claim
#   ❌ The full operational history of the battery
#   ❌ Any data beyond what is explicitly disclosed
# ══════════════════════════════════════════════════════════════════════


@dataclass
class SafetyClaim:
    """
    A single verifiable safety assertion about the battery.

    claim_id:   short identifier  e.g. "THERMAL_MAX"
    claim_text: human-readable assertion the issuer is committing to
    passed:     whether the battery passed this safety check
    severity:   "CRITICAL" | "MAJOR" | "MINOR"
                CRITICAL claims that fail block the certificate entirely.
                MAJOR failures are reported but do not block.
                MINOR failures are advisory only.
    """
    claim_id:   str
    claim_text: str
    passed:     bool
    severity:   str = "CRITICAL"

    def __post_init__(self):
        assert self.severity in ("CRITICAL", "MAJOR", "MINOR"), \
            f"severity must be CRITICAL, MAJOR or MINOR — got '{self.severity}'"


@dataclass
class ClaimCommitment:
    """
    The published commitment for a single safety claim.
    Contains the claim and its hash — but not the salt.
    The salt is retained privately by the issuer.
    """
    claim_id:        str
    claim_text:      str
    passed:          bool
    severity:        str
    commitment_hash: str    # SHA-256(claim_text + "|" + salt)
    # salt is intentionally NOT stored here — kept by issuer only


class PrivacyPreservingCert:
    """
    Issues and verifies privacy-preserving safety certificates
    for EV batteries transitioning to second-life applications.

    The certificate proves the battery passed defined safety checks
    without exposing the underlying sensor data.

    Usage:
      1. Define safety claims from your BMS data
      2. Call issue() to generate the certificate
      3. Share the certificate with the second-life operator
      4. Optionally call verify_claim() to selectively disclose
         individual claims with their salts for spot-checking
    """

    # ── Standard safety claim templates ───────────────────────────────
    # These are the claims a second-life operator typically needs.
    # Parameterise the thresholds to match your BMS specifications.
    # In production, populate passed= from real BMS sensor queries.

    @staticmethod
    def standard_claims(
        max_temp_c:            float,
        cell_voltage_variance: float,
        cycle_count:           int,
        thermal_events:        int,
        soh_pct:               float,
        # Thresholds — parameterise for your chemistry and application
        max_temp_threshold:    float = 55.0,
        max_voltage_var:       float = 0.05,
        max_cycles:            int   = 1500,
        max_thermal_events:    int   = 0,
        min_soh:               float = 75.0,
    ) -> list[SafetyClaim]:
        """
        Generate standard safety claims from BMS sensor readings.

        In production: replace the parameter values with live readings
        from your battery management system or telematics API.

        Returns a list of SafetyClaim objects ready for commitment.
        """
        return [
            SafetyClaim(
                claim_id   = "THERMAL_MAX",
                claim_text = (
                    f"Maximum recorded cell temperature did not exceed "
                    f"{max_temp_threshold}°C. "
                    f"Actual maximum: {max_temp_c}°C."
                ),
                passed   = max_temp_c <= max_temp_threshold,
                severity = "CRITICAL",
            ),
            SafetyClaim(
                claim_id   = "THERMAL_EVENTS",
                claim_text = (
                    f"No thermal runaway events recorded in operational history. "
                    f"Recorded events: {thermal_events}."
                ),
                passed   = thermal_events <= max_thermal_events,
                severity = "CRITICAL",
            ),
            SafetyClaim(
                claim_id   = "VOLTAGE_VARIANCE",
                claim_text = (
                    f"Maximum cell voltage variance within acceptable bounds "
                    f"({max_voltage_var}V). "
                    f"Actual variance: {cell_voltage_variance}V."
                ),
                passed   = cell_voltage_variance <= max_voltage_var,
                severity = "MAJOR",
            ),
            SafetyClaim(
                claim_id   = "CYCLE_COUNT",
                claim_text = (
                    f"Total charge cycles do not exceed wear limit "
                    f"({max_cycles} cycles). "
                    f"Actual cycles: {cycle_count}."
                ),
                passed   = cycle_count <= max_cycles,
                severity = "MAJOR",
            ),
            SafetyClaim(
                claim_id   = "SOH_THRESHOLD",
                claim_text = (
                    f"State of Health meets second-life minimum "
                    f"({min_soh}%). "
                    f"Actual SoH: {soh_pct}%."
                ),
                passed   = soh_pct >= min_soh,
                severity = "CRITICAL",
            ),
        ]

    def __init__(
        self,
        passport:  BatteryPassport,
        claims:    list[SafetyClaim],
        issuer_id: str = "ALA-System-v1",
    ):
        self.passport   = passport
        self.claims     = claims
        self.issuer_id  = issuer_id

        # Salts are generated once and kept private by the issuer.
        # In production: store salts in a secure vault (HSM or KMS).
        # Never publish salts alongside the certificate.
        self._salts: dict[str, str] = {
            claim.claim_id: uuid.uuid4().hex
            for claim in claims
        }

        self.issued_at_utc = datetime.now(timezone.utc).isoformat()
        self.cert_id       = str(uuid.uuid4())

        # Generate all commitments and the master certificate hash
        self.commitments   = self._generate_commitments()
        self.safe_for_reuse = self._evaluate_safety()
        self.master_hash   = self._generate_master_hash()

    # ── Internal methods ──────────────────────────────────────────────

    def _commit(self, claim: SafetyClaim) -> str:
        """
        SHA-256 commitment for a single claim.
        commitment = SHA-256(claim_text + "|" + salt)
        The "|" separator prevents concatenation attacks.
        """
        salt       = self._salts[claim.claim_id]
        preimage   = f"{claim.claim_text}|{salt}"
        return hashlib.sha256(preimage.encode("utf-8")).hexdigest()

    def _generate_commitments(self) -> list[ClaimCommitment]:
        """Generate a ClaimCommitment for every claim."""
        return [
            ClaimCommitment(
                claim_id        = c.claim_id,
                claim_text      = c.claim_text,
                passed          = c.passed,
                severity        = c.severity,
                commitment_hash = self._commit(c),
            )
            for c in self.claims
        ]

    def _evaluate_safety(self) -> bool:
        """
        Battery is safe for reuse only if all CRITICAL claims pass.
        MAJOR failures are reported but do not block the certificate.
        """
        return all(
            c.passed
            for c in self.claims
            if c.severity == "CRITICAL"
        )

    def _generate_master_hash(self) -> str:
        """
        A master SHA-256 hash over the entire certificate.
        Any change to any commitment, the passport ID, or the
        issue timestamp produces a completely different master hash.
        This seals the certificate against post-issuance tampering.
        """
        canonical = json.dumps({
            "cert_id":      self.cert_id,
            "passport_id":  self.passport.passport_id,
            "issuer_id":    self.issuer_id,
            "issued_at":    self.issued_at_utc,
            "safe_for_reuse": self.safe_for_reuse,
            "commitments":  [
                {
                    "claim_id":        c.claim_id,
                    "commitment_hash": c.commitment_hash,
                    "passed":          c.passed,
                }
                for c in self.commitments
            ],
        }, sort_keys=True, ensure_ascii=True)
        return hashlib.sha256(canonical.encode("utf-8")).hexdigest()

    # ── Public interface ──────────────────────────────────────────────

    def verify_claim(self, claim_id: str, revealed_salt: str) -> bool:
        """
        Selective disclosure verification.

        The issuer reveals (claim_text, salt) for a specific claim.
        The verifier recomputes the hash and checks it matches
        the published commitment_hash.

        Returns True if the disclosed data matches the commitment.
        Returns False if the data has been altered since issuance.

        In production: the issuer sends (claim_text, salt) through
        a secure channel. The verifier runs this method independently.
        """
        commitment = next(
            (c for c in self.commitments if c.claim_id == claim_id), None
        )
        if commitment is None:
            raise ValueError(f"No commitment found for claim_id '{claim_id}'")

        preimage      = f"{commitment.claim_text}|{revealed_salt}"
        recomputed    = hashlib.sha256(preimage.encode("utf-8")).hexdigest()
        return recomputed == commitment.commitment_hash

    def to_dict(self) -> dict:
        """
        The public certificate — contains commitments and master hash
        but never the private salts.
        """
        return {
            "cert_id":        self.cert_id,
            "passport_id":    self.passport.passport_id,
            "issuer_id":      self.issuer_id,
            "issued_at_utc":  self.issued_at_utc,
            "safe_for_reuse": self.safe_for_reuse,
            "master_hash":    self.master_hash,
            "commitments": [
                {
                    "claim_id":        c.claim_id,
                    "claim_text":      c.claim_text,
                    "passed":          c.passed,
                    "severity":        c.severity,
                    "commitment_hash": c.commitment_hash,
                }
                for c in self.commitments
            ],
            "_scheme": "SHA-256 salted commitment (privacy-preserving)",
        }

    def print_certificate(self) -> None:
        status = "✅ SAFE FOR REUSE" if self.safe_for_reuse else "❌ NOT CLEARED"
        print("━" * 62)
        print("  PRIVACY-PRESERVING SAFETY CERTIFICATE")
        print("━" * 62)
        print(f"  Certificate ID : {self.cert_id[:16]}...")
        print(f"  Passport ID    : {self.passport.passport_id[:16]}...")
        print(f"  Issued by      : {self.issuer_id}")
        print(f"  Issued at      : {self.issued_at_utc}")
        print(f"  Verdict        : {status}")
        print(f"  Master hash    : {self.master_hash[:32]}...")
        print()
        print(f"  {'Claim':<20} {'Sev':<10} {'Result'}")
        print(f"  {'─'*46}")
        for c in self.commitments:
            result = "✅ PASS" if c.passed else "❌ FAIL"
            print(f"  {c.claim_id:<20} {c.severity:<10} {result}")
            print(f"    {c.claim_text[:72]}")
            print(f"    hash: {c.commitment_hash[:32]}...")
            print()
        print("  Note: raw sensor values are NOT included in this")
        print("  certificate. Claims are proven by commitment hash only.")
        print("━" * 62)


# ── Issue the certificate for our NMC pack ────────────────────────────
# Simulated BMS readings — replace with real sensor queries in production

claims = PrivacyPreservingCert.standard_claims(
    max_temp_c            = 48.3,    # °C — highest recorded cell temp
    cell_voltage_variance = 0.031,   # V  — max variance across cells
    cycle_count           = 847,     # full charge cycles completed
    thermal_events        = 0,       # thermal runaway events
    soh_pct               = passport.circularity.current_soh_pct,
)

cert = PrivacyPreservingCert(
    passport  = passport,
    claims    = claims,
    issuer_id = "ALA-System-v1",
)

cert.print_certificate()

# ── Demonstrate selective disclosure ──────────────────────────────────
print()
print("  Selective disclosure demonstration:")
print("  Issuer reveals salt for THERMAL_MAX claim only.")
print()

# The issuer retrieves the salt from their private vault and shares it
revealed_salt = cert._salts["THERMAL_MAX"]  # In production: from secure vault
verified      = cert.verify_claim("THERMAL_MAX", revealed_salt)

print(f"  Claim     : THERMAL_MAX")
print(f"  Salt      : {revealed_salt[:16]}... (revealed by issuer)")
print(f"  Verified  : {'✅ CONFIRMED' if verified else '❌ MISMATCH'}")
print()
print("  The verifier confirmed the THERMAL_MAX claim is authentic")
print("  without seeing any other sensor data from the battery.")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  PRIVACY-PRESERVING SAFETY CERTIFICATE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Certificate ID : ed5b13fe-6027-4a...
  Passport ID    : 5d2205da-d23e-41...
  Issued by      : ALA-System-v1
  Issued at      : 2026-04-15T11:51:01.581258+00:00
  Verdict        : ✅ SAFE FOR REUSE
  Master hash    : 8fe459713029f3ddc4a6658d971e0ad4...

  Claim                Sev        Result
  ──────────────────────────────────────────────
  THERMAL_MAX          CRITICAL   ✅ PASS
    Maximum recorded cell temperature did not exceed 55.0°C. Actual maximum:
    hash: 4095b6d6a63b618dc957e33b2348f25f...

  THERMAL_EVENTS       CRITICAL   ✅ PASS
    No thermal runaway events recorded in operational history. Recorded even
    hash: 6b8ab4bfda726451e4ac0f0bb68e2a7b...

  VOLTAGE_VARIANCE     MAJOR      ✅ PASS
    Maximum cell voltage variance within acceptable bounds (0.05V). Actual v
    hash: 4fdb0c3dd59c77a63ddeb19034427b2

In [31]:
# ══════════════════════════════════════════════════════════════════════
# ALA CELL 9 — ArbitrationReport
# ══════════════════════════════════════════════════════════════════════
# The final output of the Autonomous Life-Cycle Arbitrator.
#
# Assembles all six modules into one structured, exportable document:
#   • Battery identity        (from the original passport)
#   • Dynamic carbon record   (Module 1 — DynamicCarbonTracker)
#   • Both bids               (Module 2 — RecyclerBid, SecondLifeBid)
#   • Circular scores         (Module 3 — CircularScore)
#   • Arbitration verdict     (Module 4 — LifeCycleArbitrator)
#   • Safety certificate      (Module 5 — PrivacyPreservingCert)
#
# The report is sealed with a SHA-256 hash that chains directly
# onto the original passport's combined_passport_sha256 — making
# the arbitration event part of the battery's permanent integrity trail.
#
# Every report is:
#   • Timestamped at generation
#   • Cryptographically sealed
#   • Fully exportable as JSON
#   • Human-readable via print_report()
# ══════════════════════════════════════════════════════════════════════


class ArbitrationReport:
    """
    The complete output of one arbitration event.

    One ArbitrationReport is generated each time a battery reaches
    end of primary use and negotiate_retirement() is called.
    It becomes a permanent record in the battery's digital history —
    linking the original passport to the retirement decision and the
    safety certificate that enables the next phase of life.

    In a production system, this report is:
      1. Stored in the EU Battery Passport registry
      2. Linked to the original passport via passport_id
      3. Shared with the winning bidder as their authority to proceed
      4. Archived for regulatory audit access
    """

    def __init__(
        self,
        passport:  BatteryPassport,
        tracker:   DynamicCarbonTracker,
        verdict:   RetirementVerdict,
        cert:      PrivacyPreservingCert,
        r_bid:     RecyclerBid,
        sl_bid:    SecondLifeBid,
    ):
        self.passport       = passport
        self.tracker        = tracker
        self.verdict        = verdict
        self.cert           = cert
        self.r_bid          = r_bid
        self.sl_bid         = sl_bid

        self.report_id      = str(uuid.uuid4())
        self.generated_at   = datetime.now(timezone.utc).isoformat()

        # Seal the report — chains onto the original passport hash
        self.report_hash    = self._generate_report_hash()

    # ── Integrity ─────────────────────────────────────────────────────

    def _generate_report_hash(self) -> str:
        """
        SHA-256 hash sealing the entire arbitration report.

        Chains onto the original passport by including
        combined_passport_sha256 as an input — any change to the
        original passport data invalidates this report hash,
        and vice versa. This creates a cryptographic link between
        the passport and every arbitration event in its history.
        """
        canonical = json.dumps({
            "report_id":              self.report_id,
            "generated_at":           self.generated_at,
            "passport_id":            self.passport.passport_id,
            "passport_sha256":        self.passport.integrity.combined_passport_sha256,
            "decision":               self.verdict.decision.value,
            "winner":                 self.verdict.winner_name,
            "recycler_score":         self.verdict.recycler_score.weighted_total,
            "sl_score":               self.verdict.sl_score.weighted_total,
            "cert_id":                self.cert.cert_id,
            "cert_master_hash":       self.cert.master_hash,
            "lifetime_kg_co2":        self.tracker.lifetime_kg_co2,
            "carbon_saving_pct":      self.tracker.carbon_saving_vs_new_pct,
        }, sort_keys=True, ensure_ascii=True)
        return hashlib.sha256(canonical.encode("utf-8")).hexdigest()

    # ── Export ────────────────────────────────────────────────────────

    def to_dict(self) -> dict:
        """
        Complete report as a structured dictionary.
        Ready for JSON export, database insertion, or registry upload.
        """
        return {
            # ── Report identity ───────────────────────────────────────
            "report_id":          self.report_id,
            "generated_at_utc":   self.generated_at,
            "report_hash_sha256": self.report_hash,
            "regulation":         "EU Battery Regulation 2023/1542",
            "_system":            "Autonomous Life-Cycle Arbitrator v1",

            # ── Battery identity (from original passport) ─────────────
            "battery": {
                "passport_id":              self.passport.passport_id,
                "passport_sha256":          self.passport.integrity.combined_passport_sha256,
                "model":                    self.passport.strategic.model_designation,
                "chemistry":                self.passport.strategic.chemistry.value,
                "energy_content_kwh":       self.passport.strategic.energy_content_kwh,
                "manufacturer":             self.passport.strategic.manufacturer_id,
                "manufacturing_date":       self.passport.strategic.manufacturing_date.isoformat(),
                "soh_at_arbitration_pct":   self.passport.circularity.current_soh_pct,
                "remaining_energy_kwh":     self.verdict.remaining_kwh,
            },

            # ── Dynamic carbon record (Module 1) ──────────────────────
            "carbon_record": self.tracker.to_dict(),

            # ── Bids submitted (Module 2) ─────────────────────────────
            "bids": {
                "recycler":     self.r_bid.to_dict(),
                "second_life":  self.sl_bid.to_dict(),
            },

            # ── Arbitration verdict (Modules 3 + 4) ───────────────────
            "verdict": self.verdict.to_dict(),

            # ── Safety certificate (Module 5) ─────────────────────────
            "safety_certificate": self.cert.to_dict(),
        }

    def to_json(self, indent: int = 2) -> str:
        """Export the full report as formatted JSON string."""
        return json.dumps(self.to_dict(), indent=indent, default=str)

    def save(self, filepath: str) -> None:
        """Save the report to a JSON file."""
        with open(filepath, "w", encoding="utf-8") as f:
            f.write(self.to_json())
        print(f"  Report saved → {filepath}")

    # ── Human-readable summary ────────────────────────────────────────

    def print_report(self) -> None:
        sep = "═" * 62

        print(f"\n{sep}")
        print("  AUTONOMOUS LIFE-CYCLE ARBITRATOR")
        print("  ARBITRATION REPORT")
        print(f"{sep}")
        print(f"  Report ID    : {self.report_id[:16]}...")
        print(f"  Generated at : {self.generated_at}")
        print(f"  Report hash  : {self.report_hash[:32]}...")
        print(f"  Chains onto  : {self.passport.integrity.combined_passport_sha256[:32]}...")

        print(f"\n{'─'*62}")
        print("  BATTERY IDENTITY")
        print(f"{'─'*62}")
        print(f"  Passport ID  : {self.passport.passport_id[:16]}...")
        print(f"  Model        : {self.passport.strategic.model_designation}")
        print(f"  Chemistry    : {self.passport.strategic.chemistry.value}")
        print(f"  Energy       : {self.passport.strategic.energy_content_kwh} kWh nominal")
        print(f"  SoH          : {self.passport.circularity.current_soh_pct}%")
        print(f"  Remaining    : {self.verdict.remaining_kwh} kWh usable")

        print(f"\n{'─'*62}")
        print("  LIFETIME CARBON RECORD")
        print(f"{'─'*62}")
        print(f"  Manufacturing: {self.tracker.baseline_kg_co2} kg CO2")
        print(f"  Operational  : {self.tracker.operational_kg_co2} kg CO2")
        print(f"  Lifetime     : {self.tracker.lifetime_kg_co2} kg CO2")
        print(f"  Sessions     : {len(self.tracker.sessions)} charge events")
        if self.tracker.lifetime_intensity_g_per_kwh:
            print(f"  Intensity    : {self.tracker.lifetime_intensity_g_per_kwh} gCO2/kWh")
        if self.tracker.carbon_saving_vs_new_pct is not None:
            saving = self.tracker.carbon_saving_vs_new_pct
            arrow  = "↓ greener" if saving > 0 else "↑ heavier"
            print(f"  vs new pack  : {abs(saving)}%  {arrow}")
        print("  By region:")
        for code, kg in self.tracker.carbon_by_region().items():
            print(f"    {code}  {GRID[code].name:<14} {kg:.3f} kg CO2")

        print(f"\n{'─'*62}")
        print("  BIDS RECEIVED")
        print(f"{'─'*62}")
        r_val  = self.r_bid.material_value(
            self.passport.strategic.energy_content_kwh
        )
        sl_val = self.sl_bid.bid_value(
            self.verdict.remaining_kwh,
            self.passport.circularity.current_soh_pct,
        )
        print(f"  Recycler     : {self.r_bid.bidder_name}")
        print(f"    Basis      : raw material recovery")
        print(f"    Offer      : ${r_val:.2f}")
        print(f"    Score      : {self.verdict.recycler_score.weighted_total:.4f}")
        print(f"  Second-Life  : {self.sl_bid.bidder_name}")
        print(f"    Basis      : {self.sl_bid.price_per_kwh_usd}/kWh × "
              f"{self.verdict.remaining_kwh} kWh remaining")
        if sl_val:
            print(f"    Offer      : ${sl_val:.2f}")
        else:
            print(f"    Offer      : WITHDRAWN (SoH below minimum)")
        print(f"    Score      : {self.verdict.sl_score.weighted_total:.4f}")

        print(f"\n{'─'*62}")
        print("  VERDICT")
        print(f"{'─'*62}")
        icons = {
            ArbiterDecision.SECOND_LIFE_WINS:     "♻️  SECOND LIFE",
            ArbiterDecision.RECYCLER_WINS:        "⛏️  RECYCLE",
            ArbiterDecision.SECOND_LIFE_OVERRIDE: "⚖️  SECOND LIFE (waste hierarchy override)",
            ArbiterDecision.NO_VALID_BID:         "⚠️  MANUAL REVIEW",
        }
        print(f"  Decision     : {icons[self.verdict.decision]}")
        print(f"  Awarded to   : {self.verdict.winner_name}")
        print(f"  Reasoning:")
        for i, step in enumerate(self.verdict.reasoning, 1):
            print(f"    {i}. {step}")

        print(f"\n{'─'*62}")
        print("  SAFETY CERTIFICATE")
        print(f"{'─'*62}")
        status = "✅ SAFE FOR REUSE" if self.cert.safe_for_reuse else "❌ NOT CLEARED"
        print(f"  Cert ID      : {self.cert.cert_id[:16]}...")
        print(f"  Status       : {status}")
        print(f"  Master hash  : {self.cert.master_hash[:32]}...")
        print(f"  Claims       : {len(self.cert.commitments)} evaluated")
        passed = sum(1 for c in self.cert.commitments if c.passed)
        print(f"  Passed       : {passed} / {len(self.cert.commitments)}")

        print(f"\n{'─'*62}")
        print("  INTEGRITY CHAIN")
        print(f"{'─'*62}")
        print(f"  Original passport hash:")
        print(f"    {self.passport.integrity.combined_passport_sha256}")
        print(f"  This report hash:")
        print(f"    {self.report_hash}")
        print(f"  Relationship : report_hash includes passport_sha256")
        print(f"                 as input — any change to either")
        print(f"                 invalidates the chain.")
        print(f"\n{sep}\n")


# ── Assemble and print the full report ────────────────────────────────

report = ArbitrationReport(
    passport = passport,
    tracker  = tracker,
    verdict  = verdict,
    cert     = cert,
    r_bid    = recycler_bid,
    sl_bid   = second_life_bid,
)

report.print_report()


══════════════════════════════════════════════════════════════
  AUTONOMOUS LIFE-CYCLE ARBITRATOR
  ARBITRATION REPORT
══════════════════════════════════════════════════════════════
  Report ID    : e6e0fea0-4b47-47...
  Generated at : 2026-04-15T11:51:02.187073+00:00
  Report hash  : e75f786f581a1fcb0ed72d55536463aa...
  Chains onto  : eebdedde45612843cb769586379c8a0e...

──────────────────────────────────────────────────────────────
  BATTERY IDENTITY
──────────────────────────────────────────────────────────────
  Passport ID  : 5d2205da-d23e-41...
  Model        : EV-NMC-75kWh-Gen2
  Chemistry    : LiNiMnCoO2
  Energy       : 75.0 kWh nominal
  SoH          : 90.0%
  Remaining    : 67.5 kWh usable

──────────────────────────────────────────────────────────────
  LIFETIME CARBON RECORD
──────────────────────────────────────────────────────────────
  Manufacturing: 4875.0 kg CO2
  Operational  : 371.8865 kg CO2
  Lifetime     : 5246.8865 kg CO2
  Sessions     : 20 charge events
  In

In [32]:

# ══════════════════════════════════════════════════════════════════════
# ALA CELL 10 — Full system run, export and integrity demonstration
# ══════════════════════════════════════════════════════════════════════
# This cell ties the entire system together:
#
#   1. Export the ArbitrationReport to JSON
#   2. Verify the integrity chain (passport → report)
#   3. Demonstrate tamper detection on the report hash
#   4. Print the complete system summary
#   5. Show what changes when SoH drops below the override threshold
# ══════════════════════════════════════════════════════════════════════

import os

# ── 1. Export the full report to JSON ─────────────────────────────────

output_path = "arbitration_report.json"
report.save(output_path)

# Confirm the file was written and show its size
size_kb = os.path.getsize(output_path) / 1024
print(f"  File size    : {size_kb:.1f} KB")
print(f"  Top-level keys:")
with open(output_path, "r") as f:
    data = json.load(f)
for key in data.keys():
    print(f"    {key}")

# ── 2. Verify the integrity chain ─────────────────────────────────────

print()
print("━" * 62)
print("  INTEGRITY CHAIN VERIFICATION")
print("━" * 62)

# Step 1: Verify the original passport is intact
passport_integrity = passport.verify_integrity()
print(f"  Original passport:")
print(f"    Strategic layer intact   : "
      f"{'✅' if passport_integrity['strategic_layer_intact'] else '❌'}")
print(f"    Circularity layer intact : "
      f"{'✅' if passport_integrity['circularity_layer_intact'] else '❌'}")
print(f"    Passport intact          : "
      f"{'✅ VERIFIED' if passport_integrity['passport_intact'] else '❌ TAMPERED'}")

# Step 2: Verify the report hash chains onto the passport
print()
print(f"  Report chains onto passport:")
print(f"    Passport hash  : {passport.integrity.combined_passport_sha256[:32]}...")
print(f"    Report hash    : {report.report_hash[:32]}...")

# Recompute the report hash from scratch to confirm it matches
recomputed_canonical = json.dumps({
    "report_id":          report.report_id,
    "generated_at":       report.generated_at,
    "passport_id":        passport.passport_id,
    "passport_sha256":    passport.integrity.combined_passport_sha256,
    "decision":           verdict.decision.value,
    "winner":             verdict.winner_name,
    "recycler_score":     verdict.recycler_score.weighted_total,
    "sl_score":           verdict.sl_score.weighted_total,
    "cert_id":            cert.cert_id,
    "cert_master_hash":   cert.master_hash,
    "lifetime_kg_co2":    tracker.lifetime_kg_co2,
    "carbon_saving_pct":  tracker.carbon_saving_vs_new_pct,
}, sort_keys=True, ensure_ascii=True)

recomputed_hash = hashlib.sha256(
    recomputed_canonical.encode("utf-8")
).hexdigest()

chain_intact = recomputed_hash == report.report_hash
print(f"    Chain intact   : {'✅ VERIFIED' if chain_intact else '❌ BROKEN'}")

# ── 3. Tamper demonstration on the report ─────────────────────────────

print()
print("━" * 62)
print("  TAMPER DEMONSTRATION")
print("━" * 62)
print()
print("  Simulating: someone alters the decision in the JSON file")
print("  from the real verdict to 'Recycler awarded contract'...")
print()

# Load the exported JSON and alter the decision field
with open(output_path, "r") as f:
    tampered_data = json.load(f)

original_decision = tampered_data["verdict"]["decision"]
tampered_data["verdict"]["decision"] = "Recycler awarded contract"

# Now try to verify the report hash against the tampered data
# The report hash was computed over a canonical form that included
# the original decision — altering it breaks the hash

# Recompute with the tampered decision
tampered_canonical = json.dumps({
    "report_id":          report.report_id,
    "generated_at":       report.generated_at,
    "passport_id":        passport.passport_id,
    "passport_sha256":    passport.integrity.combined_passport_sha256,
    "decision":           tampered_data["verdict"]["decision"],  # ← altered
    "winner":             verdict.winner_name,
    "recycler_score":     verdict.recycler_score.weighted_total,
    "sl_score":           verdict.sl_score.weighted_total,
    "cert_id":            cert.cert_id,
    "cert_master_hash":   cert.master_hash,
    "lifetime_kg_co2":    tracker.lifetime_kg_co2,
    "carbon_saving_pct":  tracker.carbon_saving_vs_new_pct,
}, sort_keys=True, ensure_ascii=True)

tampered_hash  = hashlib.sha256(
    tampered_canonical.encode("utf-8")
).hexdigest()
tamper_matches = tampered_hash == report.report_hash

print(f"  Original decision : {original_decision}")
print(f"  Altered decision  : {tampered_data['verdict']['decision']}")
print(f"  Original hash     : {report.report_hash[:32]}...")
print(f"  Tampered hash     : {tampered_hash[:32]}...")
print(f"  Hashes match      : {'✅ YES (not tampered)' if tamper_matches else '❌ NO — TAMPERING DETECTED'}")
print()
print("  → Changing one field in the JSON produces a completely")
print("    different SHA-256 hash. The forgery is immediately visible.")

# ── 4. What-if scenario: SoH drops below override threshold ───────────

print(f"\n{'═'*62}")
print("  AUTONOMOUS LIFE-CYCLE ARBITRATOR — SYSTEM COMPLETE")
print(f"{'═'*62}")
print()
print("  What was built:")
print()
print("  Cell 1   Imports")
print("  Cell 2   Grid Region Registry      10 regions loaded")
print("  Cell 2B  Live Grid Data            Electricity Maps API active")
print("  Cell 2C  DataValidationAgent       3-category validation suite")
print("  Cell 3   DynamicCarbonTracker      accumulates lifetime CO2")
print("  Cell 4   Charging history          20 sessions, 1307.5 kWh total")
print("  Cell 5   Bid dataclasses           RecyclerBid + SecondLifeBid")
print("           Application presets       4 use cases with RTE thresholds")
print("  Cell 6   CircularScore             4-criterion weighted model")
print("  Cell 7   LifeCycleArbitrator       2-layer decision engine")
print("           Confidence score          🟢 HIGH / 🟡 MEDIUM / 🔴 LOW")
print("  Cell 8   PrivacyPreservingCert     5 safety claims committed")
print("  Cell 9   ArbitrationReport         sealed, exportable, auditable")
print("  Cell 10  Export + verification     → arbitration_report.json")
print("  Cell 11  WhatIfEngine              scenario planning and sensitivity")
print()
print("  Key numbers from this arbitration:")
print(f"    Battery model  : {passport.strategic.model_designation}")
print(f"    Chemistry      : {passport.strategic.chemistry.value}")
print(f"    SoH            : {passport.circularity.current_soh_pct}%")
print(f"    Remaining kWh  : {verdict.remaining_kwh} kWh")
print(f"    Lifetime CO2   : {tracker.lifetime_kg_co2} kg")
print(f"    Carbon saving  : {tracker.carbon_saving_vs_new_pct}% vs new pack")
print(f"    Recycler offer : ${recycler_bid.material_value(passport.strategic.energy_content_kwh):.2f}")
sl_offer = second_life_bid.bid_value(
    verdict.remaining_kwh,
    passport.circularity.current_soh_pct,
)
print(f"    2nd-life offer : ${sl_offer:.2f}")
print(f"    Decision       : {verdict.decision.value}")
print(f"    Confidence     : {verdict.confidence_icon}  {verdict.confidence}"
      f"  (margin: {verdict.confidence_margin:.4f})")
print(f"    Winner         : {verdict.winner_name}")
print(f"    Safe for reuse : {'✅ YES' if cert.safe_for_reuse else '❌ NO'}")
print()
print("  Integrity:")
print(f"    Passport hash  : {passport.integrity.combined_passport_sha256[:32]}...")
print(f"    Report hash    : {report.report_hash[:32]}...")
print(f"    Chain intact   : {'✅ VERIFIED' if chain_intact else '❌ BROKEN'}")
print(f"    Live grid data : ✅ Electricity Maps API")
print(f"    Validation     : ✅ {validation_report.error_count} errors  "
      f"{validation_report.warning_count} warnings")
print()
print("  To plug in real data:")
print("    1. GRID intensities  → live via Electricity Maps API ✅ DONE")
print("    2. BMS charge records → replace simulated_sessions")
print("    3. LME / IRENA feeds → replace bid prices")
print("    4. Real sensor readings → replace SafetyClaim values")
print(f"\n{'═'*62}\n")

  Report saved → arbitration_report.json
  File size    : 5.2 KB
  Top-level keys:
    report_id
    generated_at_utc
    report_hash_sha256
    regulation
    _system
    battery
    carbon_record
    bids
    verdict
    safety_certificate

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  INTEGRITY CHAIN VERIFICATION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Original passport:
    Strategic layer intact   : ✅
    Circularity layer intact : ✅
    Passport intact          : ✅ VERIFIED

  Report chains onto passport:
    Passport hash  : eebdedde45612843cb769586379c8a0e...
    Report hash    : e75f786f581a1fcb0ed72d55536463aa...
    Chain intact   : ✅ VERIFIED

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  TAMPER DEMONSTRATION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  Simulating: someone alters the decision in the JSON file
  from the real verdict to 'Recycler awarded contract'...

  Original decision : Se

In [33]:
# ══════════════════════════════════════════════════════════════════════
# ALA CELL 11 — WhatIfEngine
# ══════════════════════════════════════════════════════════════════════
# Runs sensitivity analysis on a completed arbitration.
#
# Answers questions that matter to real operators:
#   → At what SoH would the second-life bid be withdrawn?
#   → If lithium price doubles, does the recycler win?
#   → Which application gets the best outcome for this battery?
#   → How much headroom does this battery have before retirement?
#
# This transforms the system from a decision engine into a
# scenario planning tool — significantly more valuable to
# operators like Circunomics and ReBattery who need to
# understand the sensitivity of decisions, not just the outcome.
# ══════════════════════════════════════════════════════════════════════


class WhatIfEngine:
    """
    Sensitivity analysis on top of a completed arbitration.

    Takes the same passport, tracker, and bids used in the main
    arbitration and runs variations to answer boundary questions.

    All analysis is read-only — the original passport, tracker,
    and verdict are never modified.
    """

    def __init__(
        self,
        passport:                 BatteryPassport,
        tracker:                  DynamicCarbonTracker,
        recycler_bid:             RecyclerBid,
        second_life_bid:          SecondLifeBid,
        weights:                  ScoringWeights = None,
        soh_sl_threshold:         float = 75.0,
        price_override_tolerance: float = 0.15,
    ):
        self.passport                 = passport
        self.tracker                  = tracker
        self.recycler_bid             = recycler_bid
        self.second_life_bid          = second_life_bid
        self.weights                  = weights or ScoringWeights()
        self.soh_sl_threshold         = soh_sl_threshold
        self.price_override_tolerance = price_override_tolerance

    def _run_arbitration(
        self,
        soh_pct:  float,
        li_price: float = None,
        co_price: float = None,
        ni_price: float = None,
    ) -> RetirementVerdict:
        """
        Run a temporary arbitration with modified parameters.
        Creates temporary objects — never modifies originals.
        """
        # Build a temporary passport with modified SoH
        temp_circularity = JTC24_CircularityLayer(
            recycled_cobalt_pct             = self.passport.circularity.recycled_cobalt_pct,
            recycled_lithium_pct            = self.passport.circularity.recycled_lithium_pct,
            recycled_nickel_pct             = self.passport.circularity.recycled_nickel_pct,
            recycled_lead_pct               = self.passport.circularity.recycled_lead_pct,
            recycled_manganese_pct          = self.passport.circularity.recycled_manganese_pct,
            carbon_footprint_kg_co2_per_kwh = self.passport.circularity.carbon_footprint_kg_co2_per_kwh,
            carbon_footprint_scope          = self.passport.circularity.carbon_footprint_scope,
            cobalt_supply_chain_verified    = self.passport.circularity.cobalt_supply_chain_verified,
            lithium_supply_chain_verified   = self.passport.circularity.lithium_supply_chain_verified,
            conflict_minerals_free          = self.passport.circularity.conflict_minerals_free,
            application                     = self.passport.circularity.application,
            dismantling_time_minutes        = self.passport.circularity.dismantling_time_minutes,
            recyclability_rate_pct          = self.passport.circularity.recyclability_rate_pct,
            recovery_efficiency_pct         = self.passport.circularity.recovery_efficiency_pct,
            lifecycle_state                 = self.passport.circularity.lifecycle_state,
            current_soh_pct                 = soh_pct,
        )
        temp_passport = BatteryPassport(self.passport.strategic, temp_circularity)

        # Build temporary recycler bid with modified prices if provided
        temp_recycler = RecyclerBid(
            bidder_id               = self.recycler_bid.bidder_id,
            bidder_name             = self.recycler_bid.bidder_name,
            li_price_per_kg         = li_price or self.recycler_bid.li_price_per_kg,
            co_price_per_kg         = co_price or self.recycler_bid.co_price_per_kg,
            ni_price_per_kg         = ni_price or self.recycler_bid.ni_price_per_kg,
            mn_price_per_kg         = self.recycler_bid.mn_price_per_kg,
            recovery_efficiency_pct = self.recycler_bid.recovery_efficiency_pct,
        )

        temp_arbitrator = LifeCycleArbitrator(
            passport                 = temp_passport,
            tracker                  = self.tracker,
            weights                  = self.weights,
            soh_sl_threshold         = self.soh_sl_threshold,
            price_override_tolerance = self.price_override_tolerance,
        )
        return temp_arbitrator.negotiate_retirement(temp_recycler, self.second_life_bid)

    # ── Analysis 1: SoH breakeven ──────────────────────────────────────

    def soh_breakeven(self) -> float:
        """
        Find the practical SoH threshold for this battery.

        For most batteries with a strong second-life price advantage
        the decision never flips on score alone within the operational
        range. The real threshold is the operator minimum SoH —
        below which the bid is formally withdrawn.

        Returns the operator minimum SoH as the effective breakeven.
        """
        return self.second_life_bid.min_soh_pct

    def print_soh_breakeven(self) -> None:
        current      = self.passport.circularity.current_soh_pct
        operator_min = self.second_life_bid.min_soh_pct
        headroom     = round(current - operator_min, 2)
        remaining    = self.passport.assess_second_life()["remaining_energy_kwh"]

        print("━" * 62)
        print("  WHAT-IF: SoH BREAKEVEN ANALYSIS")
        print("━" * 62)
        print(f"  Current SoH           : {current}%")
        print(f"  Operator minimum SoH  : {operator_min}%")
        print(f"  Headroom              : {headroom}% before bid withdrawn")
        print()
        print(f"  Application           : {self.second_life_bid.target_application}")
        print()

        if headroom > 20:
            print("  🟢 VERY HIGH headroom — battery has significant life remaining")
            print("     Second life wins strongly at all viable degradation levels")
        elif headroom > 10:
            print("  🟢 HIGH headroom — battery well within second-life range")
        elif headroom > 5:
            print("  🟡 MEDIUM headroom — monitor degradation closely")
        else:
            print("  🔴 LOW headroom — approaching bid withdrawal threshold")

        print()
        print(f"  Note: For this battery the second-life price advantage")
        print(f"  (${self.second_life_bid.price_per_kwh_usd}/kWh × "
              f"{remaining} kWh remaining)")
        print(f"  is so strong that the recycler cannot win on Circular Score")
        print(f"  alone. Second life only loses if SoH drops below the")
        print(f"  operator minimum of {operator_min}% and the bid is withdrawn.")
        print("━" * 62)

    # ── Analysis 2: Price sensitivity ─────────────────────────────────

    def price_sensitivity(
        self,
        li_multipliers: list[float] = None,
    ) -> list[dict]:
        """
        Show how the decision changes as lithium price varies.

        li_multipliers: list of price multipliers to test.
          e.g. [0.5, 1.0, 1.5, 2.0, 3.0]
          means: test at 50%, 100%, 150%, 200%, 300% of current price.

        Returns a list of results showing decision at each price point.
        This answers: at what lithium price does the recycler start winning?
        """
        if li_multipliers is None:
            li_multipliers = [0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 5.0]

        base_price = self.recycler_bid.li_price_per_kg
        results    = []

        for multiplier in li_multipliers:
            new_price = round(base_price * multiplier, 2)
            verdict   = self._run_arbitration(
                soh_pct  = self.passport.circularity.current_soh_pct,
                li_price = new_price,
            )
            recycler_offer = RecyclerBid(
                bidder_id               = self.recycler_bid.bidder_id,
                bidder_name             = self.recycler_bid.bidder_name,
                li_price_per_kg         = new_price,
                co_price_per_kg         = self.recycler_bid.co_price_per_kg,
                ni_price_per_kg         = self.recycler_bid.ni_price_per_kg,
                mn_price_per_kg         = self.recycler_bid.mn_price_per_kg,
                recovery_efficiency_pct = self.recycler_bid.recovery_efficiency_pct,
            ).material_value(self.passport.strategic.energy_content_kwh)

            results.append({
                "li_multiplier":   multiplier,
                "li_price_per_kg": new_price,
                "recycler_offer":  recycler_offer,
                "decision":        verdict.decision.value,
                "winner":          verdict.winner_name,
                "confidence":      verdict.confidence,
            })

        return results

    def print_price_sensitivity(self) -> None:
        results = self.price_sensitivity()
        base    = self.recycler_bid.li_price_per_kg

        print("━" * 62)
        print("  WHAT-IF: LITHIUM PRICE SENSITIVITY")
        print("━" * 62)
        print(f"  Base lithium price: ${base}/kg")
        print()
        print(f"  {'Multiplier':<12} {'Li $/kg':<10} "
              f"{'Recycler $':<14} {'Decision':<35} {'Conf'}")
        print(f"  {'─'*60}")

        for r in results:
            flip     = "← FLIP" if r["decision"] == ArbiterDecision.RECYCLER_WINS.value else ""
            decision = r["decision"][:32]
            print(f"  {r['li_multiplier']:<12.1f}x "
                  f"${r['li_price_per_kg']:<9.2f} "
                  f"${r['recycler_offer']:<13.2f} "
                  f"{decision:<35} "
                  f"{r['confidence']:<8} {flip}")

        print("━" * 62)

    # ── Analysis 3: Application comparison ────────────────────────────

    def application_comparison(self) -> list[dict]:
        """
        Test this battery against all four application presets.

        Shows which applications this battery qualifies for and
        which would withdraw their bid based on SoH.

        This answers: what is the best application for this specific
        battery given its current condition?
        """
        soh     = self.passport.circularity.current_soh_pct
        results = []

        for app_name, preset in SecondLifeBid.APPLICATION_PRESETS.items():
            temp_bid = SecondLifeBid.for_application(
                application       = app_name,
                bidder_id         = "test_operator",
                bidder_name       = f"Test Operator ({app_name})",
                price_per_kwh_usd = self.second_life_bid.price_per_kwh_usd,
            )

            remaining = self.passport.assess_second_life()["remaining_energy_kwh"]
            bid_val   = temp_bid.bid_value(remaining, soh)

            results.append({
                "application":  app_name,
                "description":  preset["description"],
                "min_soh_pct":  preset["min_soh_pct"],
                "min_rte_pct":  preset["min_rte_pct"],
                "qualifies":    bid_val is not None,
                "bid_value":    bid_val,
            })

        return results

    def print_application_comparison(self) -> None:
        results = self.application_comparison()

        print("━" * 62)
        print("  WHAT-IF: APPLICATION COMPARISON")
        print("━" * 62)
        print(f"  Battery SoH: {self.passport.circularity.current_soh_pct}%")
        print()

        for r in results:
            status = "✅ QUALIFIES" if r["qualifies"] else "❌ WITHDRAWN"
            val    = f"${r['bid_value']:.2f}" if r["bid_value"] else "—"
            print(f"  {status}  {r['application']}")
            print(f"    Min SoH: {r['min_soh_pct']}%  "
                  f"Min RTE: {r['min_rte_pct']}%  "
                  f"Bid value: {val}")
            print(f"    {r['description']}")
            print()

        print("━" * 62)


# ── Run all three analyses ─────────────────────────────────────────────

what_if = WhatIfEngine(
    passport                 = passport,
    tracker                  = tracker,
    recycler_bid             = recycler_bid,
    second_life_bid          = second_life_bid,
    soh_sl_threshold         = 75.0,
    price_override_tolerance = 0.15,
)

print()
what_if.print_soh_breakeven()
print()
what_if.print_price_sensitivity()
print()
what_if.print_application_comparison()


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  WHAT-IF: SoH BREAKEVEN ANALYSIS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Current SoH           : 90.0%
  Operator minimum SoH  : 72.0%
  Headroom              : 18.0% before bid withdrawn

  Application           : solar_buffer_storage

  🟢 HIGH headroom — battery well within second-life range

  Note: For this battery the second-life price advantage
  ($78.0/kWh × 67.5 kWh remaining)
  is so strong that the recycler cannot win on Circular Score
  alone. Second life only loses if SoH drops below the
  operator minimum of 72.0% and the bid is withdrawn.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  WHAT-IF: LITHIUM PRICE SENSITIVITY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Base lithium price: $13.0/kg

  Multiplier   Li $/kg    Recycler $     Decision                            Conf
  ────────